# IMPORTS

In [ ]:
import torch
import numpy as np
import pandas as pd
import time
from PIL import Image
import shutil
import random
import torch
from torchvision import transforms
from facenet_pytorch import InceptionResnetV1
import tensorflow as tf
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import CarliniLInfMethod
import tarfile
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
import shutil
import random
import time
import pickle

# Data and Machine Learning
import pandas as pd
import numpy as np
import tensorflow as tf

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import torchvision.models as models
from models.cydonia_resnet import resnet50
import torch.nn.functional as F

# Adversarial Robustness Toolbox (ART)
import art
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import ProjectedGradientDescentPyTorch
from art.attacks.evasion import BasicIterativeMethod
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import CarliniLInfMethod
from art.defences.preprocessor import SpatialSmoothing, JpegCompression

print(f"TensorFlow version: {tf.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"ART version: {art.__version__}")
print("All libraries successfully imported.\n")

# CONFIGURATIONS

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")


# ========================================
# ATTACK CONFIGURATIONS
# ========================================
WORKER = "Salvatore"
BATCH_SIZE  = 64

# ========================================
# DIRECTORIES AND PATHS CONFIGURATIONS
# ========================================
# Base project directories
DATA_DIR = './dataset' 
MODELS_DIR   = './models'
RESULTS_DIR = './results'
ADVERSARIAL_DATA_DIR = './adversarial_data'                                    # Base directory for all adversarial examples generated in the project
DATA_DIR_100 = os.path.join(DATA_DIR, 'dataset_1_per_person')                  # Dataset dir for DF and Carlini Wagner with 100 images (1 per subject)
DATA_DIR_1000 = os.path.join(DATA_DIR, 'dataset_100_subjects')                 # Dataset dir for PGD, FGSM and BIM attacks with 1000 images (10 per subject)
CSV_PATH = os.path.join(DATA_DIR, 'selected_100_subjects.csv')                 # CSV file with metadata for the 100 subjects used in the dataset
BASE_ADVERSARIAL_CW_DIR = os.path.join(ADVERSARIAL_DATA_DIR, 'CW_Untargeted')  # Directory for Carlini Wagner untargeted adversarial examples
OUTPUT_CSV_CW = "./results/final_cw_results_with_accuracy.csv"
CSV_CANDIDO = "cw_grid_search_worker_Candido.csv"
CSV_CHRISTIAN = "cw_grid_search_worker_Christian.csv"


# ========================================
# FLAG CONFIGURATIONS
# ========================================
# Configuration flag for baseline evaluation
RUN_BASELINE_EVAL = True
# Configuration flags for adversarial attacks
RUN_FGSM_GENERIC    = False
RUN_PGD_GENERIC     = False
RUN_BIM_GENERIC     = False
RUN_CW_GENERIC      = False
RUN_FGSM_SPECIFIC   = False
RUN_BIM_SPECIFIC    = True
RUN_PGD_SPECIFIC    = True
RUN_CW_SPECIFIC     = False
# Configuration flags for transferability evaluation
RUN_FGSM_TRANSFERABILITY_GENERIC    = True
RUN_BIM_TRANSFERABILITY_GENERIC     = True
RUN_PGD_TRANSFERABILITY_GENERIC     = True
RUN_CW_TRANSFERABILITY_GENERIC      = True
RUN_FGSM_TRANSFERABILITY_SPECIFIC   = True
RUN_BIM_TRANSFERABILITY_SPECIFIC    = True
RUN_PGD_TRANSFERABILITY_SPECIFIC    = True
RUN_CW_TRANSFERABILITY_SPECIFIC     = True


# ========================================
# MODELS INITIALIZATIONS
# ========================================
print("\nLoading InceptionResnetV1 (NN1)")
nn1 = InceptionResnetV1(pretrained='vggface2').eval().to(device)
nn1.classify = True

print("\nLoading ResNet-50 (NN2)")
NN2_WEIGHT_PATH = os.path.join(MODELS_DIR, 'resnet50_ft_weight.pkl')
if not os.path.exists(NN2_WEIGHT_PATH):
    raise FileNotFoundError(
        f"{NN2_WEIGHT_PATH} Not Found: Please download the ResNet-50 weights from "
        "https://drive.google.com/file/d/1A94PAAnwk6L7hXdBXLFosB_s0SzEhAFU/view "
        "and place them in the ./models/ directory.\n"
    )

# 1. Initialize the custom ResNet-50 architecture.
nn2 = resnet50(num_classes=8631)                                             
# 2. Read the raw binary data using standard file operations
with open(NN2_WEIGHT_PATH, 'rb') as f:
    obj = f.read()
    # The file was serialized in Python 2; encoding='latin1' is mandatory for Python 3 compatibility
    raw_weights = pickle.loads(obj, encoding='latin1')
# 3. Convert NumPy arrays to PyTorch tensors and clean the dictionary keys
clean_state_dict = {}
for k, v in raw_weights.items():
    # Remove the "module." prefix in case the original model used DataParallel
    clean_key = k.replace("module.", "")
    # Convert the extracted NumPy array into a PyTorch tensor
    clean_state_dict[clean_key] = torch.from_numpy(v)
# 4. Load the properly formatted weights into the model
nn2.load_state_dict(clean_state_dict)
# 5. Set the model to evaluation mode and move to the target device
nn2 = nn2.eval().to(device)
print("NN2 (ResNet-50) weights loaded successfully!")


# Load official labels
fpath = tf.keras.utils.get_file(
    fname='rcmalli_vggface_labels_v2.npy',
    origin="https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_labels_v2.npy",
    cache_dir=str(MODELS_DIR),
    cache_subdir='.'
)
LABELS = np.load(fpath, allow_pickle=True)
print(f"Found {len(LABELS)} classes.")

# Dataset Preparation

This step verifies whether the datasets have already been prepared. If the extracted
dataset folders and the subject CSV are found locally, the step is skipped entirely.

Two datasets are prepared:
- `dataset_100_subjects`: 10 images per subject (1000 total), used for FGSM, BIM, and PGD.
- `dataset_1_per_person`: 1 image per subject (100 total), used for DeepFool and Carlini-Wagner,
  which are significantly slower and cannot be run on the full 1000-image dataset.

If neither dataset is found, the metadata CSV is read to select 100 subjects from the
VGGFace2 training set, images are extracted from the local TAR archive, and the
1-per-person subset is derived from the extracted data.



In [ ]:
# ==============================================================================
#  Dataset Preparation
# ==============================================================================

print("Starting Dataset Preparation...")

# ------------------------------------------------------------------------------
# 1.0 Check if both datasets are already prepared
# ------------------------------------------------------------------------------
if os.path.exists(CSV_PATH) and os.path.exists(DATA_DIR_1000) and os.path.exists(DATA_DIR_100):
    selected_subjects = pd.read_csv(CSV_PATH)
    print(f"All datasets already prepared. Loaded {len(selected_subjects)} subjects from existing CSV.")

else:
    # --------------------------------------------------------------------------
    # 1.1 Process Metadata CSV and select 100 subjects
    # --------------------------------------------------------------------------
    if not os.path.exists(CSV_PATH):
        print("Metadata CSV not found. Processing VGGFace2 metadata...")

        LOCAL_META_CSV = os.path.join(DATA_DIR, 'vggface2_meta.csv')
        if not os.path.exists(LOCAL_META_CSV):
            raise FileNotFoundError(
                f"Metadata CSV not found at {LOCAL_META_CSV}. "
                "Place vggface2_meta.csv in the dataset directory before running this step."
            )

        df = pd.read_csv(LOCAL_META_CSV, on_bad_lines='skip', skipinitialspace=True)
        df.columns = df.columns.str.strip()

        if 'Sample_Num' not in df.columns:
            df.columns = ['Class_ID', 'Name', 'Sample_Num', 'Flag', 'Gender']

        df['Sample_Num'] = pd.to_numeric(df['Sample_Num'], errors='coerce')
        df['Flag']       = pd.to_numeric(df['Flag'],       errors='coerce')

        df_filtered = df[(df['Sample_Num'] >= 10) & (df['Flag'] == 1)].copy()
        df_filtered.loc[:, 'Class_ID'] = df_filtered['Class_ID'].astype(str).str.strip()

        selected_subjects = df_filtered.sample(n=100, random_state=42)
        target_ids = {class_id: 0 for class_id in selected_subjects['Class_ID']}

        # Normalize commas in Name column to match the official VGGFace2 label format
        selected_subjects['Name'] = (selected_subjects['Name']
                                    .str.replace(',_', '_', regex=False)
                                    .str.replace(',',  '_', regex=False))

        selected_subjects.to_csv(CSV_PATH, index=False)
        print(f"Saved metadata for 100 selected subjects to: {CSV_PATH}")
    else:
        selected_subjects = pd.read_csv(CSV_PATH)
        target_ids = {class_id: 0 for class_id in selected_subjects['Class_ID']}
        print(f"Loaded {len(selected_subjects)} subjects from existing CSV.")

    # --------------------------------------------------------------------------
    # 1.2 Extract 10 images per subject from the TAR archive (1000 images total)
    # --------------------------------------------------------------------------
    if not os.path.exists(DATA_DIR_1000):
        os.makedirs(DATA_DIR_1000, exist_ok=True)

        LOCAL_TAR_PATH = os.path.join(DATA_DIR, 'vggface2.tar.gz')
        if not os.path.exists(LOCAL_TAR_PATH):
            raise FileNotFoundError(
                f"TAR archive not found at {LOCAL_TAR_PATH}. "
                "Place vggface2.tar.gz in the dataset directory before running this step."
            )
        print("\nStarting extraction of 10 images per subject...")
        IMAGES_PER_SUBJECT = 10
        total_needed       = len(target_ids) * IMAGES_PER_SUBJECT
        total_extracted    = 0
        try:
            with tarfile.open(LOCAL_TAR_PATH, 'r:*') as tar:
                for member in tar.getmembers():
                    if total_extracted >= total_needed:
                        print(f"\nExtraction complete: {total_extracted} images collected.")
                        break

                    if member.isfile():
                        parts = member.name.split('/')
                        if len(parts) >= 2 and parts[-1].lower().endswith(('.jpg', '.jpeg', '.png')):
                            class_id = parts[-2].strip()

                            if class_id in target_ids and target_ids[class_id] < IMAGES_PER_SUBJECT:
                                dest_folder = os.path.join(DATA_DIR_1000, class_id)
                                os.makedirs(dest_folder, exist_ok=True)

                                extracted_f = tar.extractfile(member)
                                if extracted_f is not None:
                                    with open(os.path.join(dest_folder, parts[-1]), 'wb') as out:
                                        shutil.copyfileobj(extracted_f, out)

                                    target_ids[class_id] += 1
                                    total_extracted      += 1

                                    if total_extracted % 100 == 0:
                                        print(f"  Progress: {total_extracted}/{total_needed} images extracted...")

        except Exception as e:
            print(f"\nAn error occurred during extraction: {e}")

        print("10-images-per-subject dataset preparation completed.")
    else:
        print(f"Dataset '{DATA_DIR_1000}' already exists. Skipping extraction.")

    # --------------------------------------------------------------------------
    # 1.3 Build 1-per-person subset from the extracted dataset
    # --------------------------------------------------------------------------
    if not os.path.exists(DATA_DIR_100):
        os.makedirs(DATA_DIR_100, exist_ok=True)

        print("\nBuilding 1-per-person subset for DeepFool and Carlini-Wagner attacks...")

        rng             = random.Random(42)
        processed_count = 0

        for subject_folder in os.listdir(DATA_DIR_1000):
            subject_path = os.path.join(DATA_DIR_1000, subject_folder)

            if os.path.isdir(subject_path):
                images = [
                    f for f in os.listdir(subject_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ]

                if images:
                    selected_image = rng.choice(images)
                    ext            = os.path.splitext(selected_image)[1]
                    new_filename   = f"{subject_folder}{ext}"

                    shutil.copy2(
                        os.path.join(subject_path, selected_image),
                        os.path.join(DATA_DIR_100, new_filename)
                    )
                    processed_count += 1

        print(f"1-per-person subset completed: {processed_count} images saved to {DATA_DIR_100}.")
    else:
        print(f"Dataset '{DATA_DIR_100}' already exists. Skipping subset creation.")

print("\nDataset preparation completed.")

## Adversarial Attacks Setup and Helper Functions

This step initializes all shared components required by the adversarial attack pipeline:
the ART classifier wrapper around NN1, the attack preprocessing transform, and three
helper functions used by every subsequent attack step (image saving, Security Evaluation
Curve plotting, and adversarial archive management).


In [ ]:
# ==============================================================================
#  Adversarial Attacks Setup and Helper Functions
# ==============================================================================

print("Initializing adversarial attack setup and helper functions...")

# ------------------------------------------------------------------------------
# 3.0.1 ART Classifier Wrapper
# ------------------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

# ART expects inputs in [0, 1] and handles the normalization shift to [-1, 1] internally.
art_classifier = PyTorchClassifier(
    model=nn1,
    clip_values=(0.0, 1.0),
    loss=criterion,
    optimizer=None,
    input_shape=(3, 160, 160),
    nb_classes=8631,
    preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
    device_type='gpu' if torch.cuda.is_available() else 'cpu'
)

# Preprocessing for attack generation: only resize and ToTensor.
# Normalization to [-1, 1] is delegated to the ART wrapper above.
preprocess_attack = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

# ------------------------------------------------------------------------------
# Helper 1: Label Normalization
# ------------------------------------------------------------------------------
def normalize_label(label: str) -> str:
    """
    Normalizes a subject name string for safe comparison against official VGGFace2 labels.
    Converts to lowercase, replaces spaces with underscores, and removes commas.
    """
    return label.lower().replace(" ", "_").replace(",_", "_").replace(",", "_")

# ------------------------------------------------------------------------------
# Helper 2: Image Saving
# ------------------------------------------------------------------------------
def save_adversarial_images(adv_examples, true_labels, filenames, attack_name, epsilon):
    """
    Saves adversarial examples generated by ART as PNG image files.

    ART outputs arrays in the [0.0, 1.0] range due to clip_values=(0.0, 1.0).
    Each image is scaled to [0, 255], transposed from (C, H, W) to (H, W, C),
    and saved under ADVERSARIAL_DATA_DIR//eps_//.png.
    """
    eps_folder_name = f"eps_{epsilon:.3f}".replace('.', '_')
    save_dir = os.path.join(ADVERSARIAL_DATA_DIR, attack_name, eps_folder_name)

    for i in range(len(adv_examples)):
        class_id   = str(true_labels[i]).strip()
        base_name  = os.path.splitext(str(filenames[i]))[0]
        subject_dir = os.path.join(save_dir, class_id)
        os.makedirs(subject_dir, exist_ok=True)

        img_array = (adv_examples[i] * 255).astype(np.uint8)
        img_array = np.transpose(img_array, (1, 2, 0))
        Image.fromarray(img_array, mode='RGB').save(
            os.path.join(subject_dir, f"{base_name}.png")
        )

# ------------------------------------------------------------------------------
# Helper 3: Security Evaluation Curve
# ------------------------------------------------------------------------------
def plot_security_evaluation_curve(epsilons, accuracies, attack_name, save_dir, baseline_acc):
    """
    Generates and saves a Security Evaluation Curve (SEC) for a given attack.

    The curve plots model accuracy as a function of the perturbation budget epsilon,
    connecting the clean baseline point (epsilon=0) to the first attack measurement
    and continuing through all evaluated epsilon values.
    """
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plt.plot([0.0, epsilons[0]], [baseline_acc, accuracies[0]],
            linestyle='-', color='#ff7f0e', linewidth=1.5)
    plt.plot(epsilons, accuracies,
            marker='o', linestyle='-', color='#ff7f0e',
            linewidth=1.5, markersize=6, label=f'NN1 Performance ({attack_name})')
    plt.plot(0.0, baseline_acc,
            marker='o', markerfacecolor='#ff7f0e', markeredgecolor='#2ca02c',
            markeredgewidth=1, markersize=6, linestyle='None',
            label=f'Baseline Accuracy ({baseline_acc:.2f}%)')

    plt.title(f'Security Evaluation Curve - {attack_name}', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel(r'Perturbation Budget ($\epsilon$ - $L_\infty$ Norm)', fontsize=14)
    plt.ylabel('Model Accuracy (%)', fontsize=14)
    plt.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    plt.xticks(np.arange(0.0, 0.11, 0.01), fontsize=12)
    plt.yticks(np.arange(0, 101, 10), fontsize=12)
    plt.ylim(-5, 105)
    plt.xlim(-0.005, 0.105)
    plt.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
    plt.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] SEC graph saved at: {plot_path}")
    
    
    
def plot_iteration_evaluation_curve(iterations, accuracies, attack_name, save_dir, fixed_eps):
    """
    Generates and saves a curve plotting model accuracy against the number of PGD iterations.
    """
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(9, 6))

    plt.plot(iterations, accuracies,
            marker='s', linestyle='-', color='#d62728',
            linewidth=1.5, markersize=6, label=f'NN1 Performance ({attack_name})')

    plt.title(f'Iteration Impact Curve - {attack_name} (Fixed $\epsilon$ = {fixed_eps})', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Number of Iterations (max_iter)', fontsize=14)
    plt.ylabel('Model Accuracy (%)', fontsize=14)
    plt.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    plt.xticks(iterations, fontsize=12)
    plt.yticks(np.arange(0, 101, 10), fontsize=12)
    plt.ylim(-5, 105)
    plt.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_Iteration_Impact.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n[INFO] Iteration impact graph saved at: {plot_path}")

print("Adversarial attack setup completed successfully.")



# ------------------------------------------------------------------------------
# Helper 4: Least Likely Class Index
# ------------------------------------------------------------------------------
def get_least_likely_indices(x_batch_clean):
    """
    Returns the least likely class index for each image in the batch,
    defined as the class assigned the lowest softmax probability by the
    clean model. Used as the target for error-specific least-likely attacks.
    """
    preds = art_classifier.predict(x_batch_clean)
    return np.argmin(preds, axis=1).tolist()


# ------------------------------------------------------------------------------
# Helper 5: Targeted Security Evaluation Curve
# ------------------------------------------------------------------------------
def plot_targeted_sec(epsilons, attack_success_rates, attack_name, save_dir):
    """
    Generates and saves a Targeted Security Evaluation Curve (SEC).

    Unlike the standard SEC which plots accuracy (decreasing with epsilon),
    the targeted SEC plots Attack Success Rate (ASR), which increases with
    epsilon as the attack becomes more capable of forcing the target class.
    """
    os.makedirs(save_dir, exist_ok=True)

    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot([0.0, epsilons[0]], [0.0, attack_success_rates[0]],
            linestyle='-', color='#d62728', linewidth=1.5)
    ax.plot(epsilons, attack_success_rates,
            marker='o', linestyle='-', color='#d62728',
            linewidth=1.5, markersize=6,
            label=f'Attack Success Rate ({attack_name})')
    ax.plot(0.0, 0.0,
            marker='o', markerfacecolor='#d62728',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label='Baseline ASR (~0%)')

    ax.set_title(f'Targeted Security Evaluation Curve — {attack_name}', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper left', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(save_dir, f'{attack_name}_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Targeted SEC graph saved at: {plot_path}")



def clean_pred_label(pred_idx):
    """
    Helper function to parse output label arrays uniformly across 
    different formats and return a clean string label.
    """
    raw_l = LABELS[pred_idx]
    if isinstance(raw_l, np.ndarray):
        raw_l = raw_l.flatten()[0]
    if isinstance(raw_l, bytes):
        raw_l = raw_l.decode('utf-8')
    return str(raw_l).replace("b'", "").replace("'", "").replace("[", "").replace("]", "").strip().lower().replace(" ", "_").replace(",_", "_").replace(",", "_")


## NN1 & NN2 Baseline Evaluation

This step loads the target model NN1 (InceptionResnetV1 pretrained on VGGFace2) and NN2 (resnet50)
and evaluates its accuracy on the 1000-image clean test set. The resulting baseline
accuracy is the reference point against which all adversarial attacks will be measured.

In [ ]:
if RUN_BASELINE_EVAL:
    # ------------------------------------------------------------------------------
    #  Step 2: Baseline Evaluation on Both Datasets
    # ------------------------------------------------------------------------------
    # ------------------------------------------------------------------------------
    # 2.1 Load Metadata Mapping (ID to Name)
    # ------------------------------------------------------------------------------
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Metadata CSV not found at {CSV_PATH}. Run Step 1 first.")

    selected_subjects = pd.read_csv(CSV_PATH)
    id_to_name = dict(zip(
        selected_subjects['Class_ID'].astype(str).str.strip(),
        selected_subjects['Name'].astype(str).str.strip()
    ))
    print("Metadata loaded for label matching.")

    # ------------------------------------------------------------------------------
    # 2.2 Define Preprocessing
    # ------------------------------------------------------------------------------
    preprocess_nn1 = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    # Nearest Neighbor interpolation preserves the exact values ​​of the original pixels without averaging, 
    # thus better preserving the L∞ perturbation. It is the most correct approach when you want to measure 
    # pure transferability without artificial attenuation.
    preprocess_nn2 = transforms.Compose([
        transforms.Resize(288, interpolation=transforms.InterpolationMode.NEAREST),                  # Resize shortest side to 288 with Nearest Neighbor interpolation
        transforms.CenterCrop(224),              # Center crop to 224x224
        transforms.ToTensor(),                   # [0, 255] -> [0.0, 1.0], shape (C, H, W)
        transforms.Lambda(lambda x: x * 255.0), # Rescale back to [0, 255]
        transforms.Lambda(lambda x: x[[2, 1, 0], :, :]),  # RGB -> BGR
        transforms.Normalize(
            mean=[91.4953, 103.8827, 131.0912],  # BGR mean subtraction (training values)
            std=[1.0, 1.0, 1.0]                  # No std scaling — pure mean subtraction
        )
    ])

    # ------------------------------------------------------------------------------
    # 2.3 Dataset Configurations Definitions
    # ------------------------------------------------------------------------------
    # Define targets to evaluate both datasets sequentially
    datasets_to_evaluate = [
        {"name": "DATASET_1000", "path": DATA_DIR_1000, "structure": "nested"},
        {"name": "DATASET_100",  "path": DATA_DIR_100,  "structure": "flat"}
    ]
    
    baseline_results = {}

    for target_ds in datasets_to_evaluate:
        ds_name = target_ds["name"]
        ds_path = target_ds["path"]
        ds_structure = target_ds["structure"]

        if not os.path.exists(ds_path):
            print(f"\n[WARNING] Path for {ds_name} does not exist at {ds_path}. Skipping evaluation.")
            continue

        current_image_paths = []
        current_true_ids = []

        # --------------------------------------------------------------------------
        # 2.3.1 Build Image List based on directory structure
        # --------------------------------------------------------------------------
        if ds_structure == "nested":
            # For 1000 images dataset (Subject Folders -> Images)
            for class_id in os.listdir(ds_path):
                class_folder = os.path.join(ds_path, class_id)
                if not os.path.isdir(class_folder):
                    continue
                for img_name in os.listdir(class_folder):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        current_image_paths.append(os.path.join(class_folder, img_name))
                        current_true_ids.append(class_id.strip())
        
        elif ds_structure == "flat":
            # For 100 images dataset (Flat Folder -> ClassID.jpg)
            for img_name in os.listdir(ds_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    current_image_paths.append(os.path.join(ds_path, img_name))
                    # Extract class target index from the filename (e.g., 'n001234.jpg' -> 'n001234')
                    class_id = os.path.splitext(img_name)[0]
                    current_true_ids.append(class_id.strip())

        # --------------------------------------------------------------------------
        # 2.4 Evaluate Current Dataset Baseline Accuracy
        # --------------------------------------------------------------------------
        print(f"\nEvaluating baseline accuracy on {ds_name} ({len(current_image_paths)} images)...")
        correct_preds_nn1 = 0
        correct_preds_nn2 = 0

        with tqdm(total=len(current_image_paths), desc=f"Evaluating {ds_name}", unit="img") as pbar:
            for img_path, true_id in zip(current_image_paths, current_true_ids):
                try:
                    img        = Image.open(img_path).convert('RGB')
                    img_tensor_nn1 = preprocess_nn1(img).unsqueeze(0).to(device)
                    img_tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)
                    

                    with torch.no_grad():
                        # NN1 Evaluation
                        preds_1    = nn1(img_tensor_nn1)
                        pred_idx_1 = np.array(preds_1[0].detach().cpu().numpy()).argmax()
                        # NN2 Evaluation 
                        preds_2    = nn2(img_tensor_nn2)
                        pred_idx_2 = np.array(preds_2[0].detach().cpu().numpy()).argmax()

                    # Resolve True Label Mapping
                    true_name  = id_to_name.get(true_id, "Unknown")
                    true_clean = true_name.lower().replace(" ", "_").replace(",_", "_").replace(",", "_")

                    pred_clean_1 = clean_pred_label(pred_idx_1)
                    pred_clean_2 = clean_pred_label(pred_idx_2)

                    if pred_clean_1 == true_clean:
                        correct_preds_nn1 += 1
                        
                    if pred_clean_2 == true_clean:
                        correct_preds_nn2 += 1

                except Exception as e:
                    print(f"Error processing {img_path}: {e}")

                pbar.update(1)

        total_images   = len(current_image_paths)
        clean_accuracy_nn1      = (correct_preds_nn1 / total_images) * 100 if total_images > 0 else 0.0
        clean_accuracy_nn2      = (correct_preds_nn2 / total_images) * 100 if total_images > 0 else 0.0
        
        # Save results into temporary tracking dictionary
        baseline_results[ds_name] = {
            "total":       total_images,
            "nn1_acc":     clean_accuracy_nn1,
            "nn1_correct": correct_preds_nn1,
            "nn2_acc":     clean_accuracy_nn2,
            "nn2_correct": correct_preds_nn2,
            "paths":       current_image_paths,
            "ids":         current_true_ids
        }

    # ------------------------------------------------------------------------------
    # 2.6 Assign Global Variables and Print Reports
    # ------------------------------------------------------------------------------
    # Extract specific values for each global placeholder variable safely
    BASELINE_ACC_1000 = baseline_results.get("DATASET_1000", {}).get("nn1_acc", 0.0)
    BASELINE_ACC_100  = baseline_results.get("DATASET_100", {}).get("nn1_acc", 0.0)
    
    BASELINE_ACC_NN2_1000 = baseline_results.get("DATASET_1000", {}).get("nn2_acc", 0.0)
    BASELINE_ACC_NN2_100  = baseline_results.get("DATASET_100", {}).get("nn2_acc", 0.0)

    # Legacy mapping to keep code backwards compatible with subsequent stages
    BASELINE_ACC    = BASELINE_ACC_1000
    image_paths     = baseline_results.get("DATASET_1000", {}).get("paths", [])
    true_folder_ids = baseline_results.get("DATASET_1000", {}).get("ids", [])

    print("\n" + "=" * 60)
    print("NN1 & NN2 - GLOBAL BASELINE EVALUATION REPORT")
    print("=" * 60)
    
    if "DATASET_1000" in baseline_results:
        res = baseline_results['DATASET_1000']
        print(f"Dataset 1000 (Nested):")
        print(f"  -> NN1 Accuracy: {res['nn1_acc']:.2f}% ({res['nn1_correct']}/{res['total']})")
        print(f"  -> NN2 Accuracy: {res['nn2_acc']:.2f}% ({res['nn2_correct']}/{res['total']})")
        
    if "DATASET_100" in baseline_results:
        res = baseline_results['DATASET_100']
        print(f"\nDataset 100 (Flat):")
        print(f"  -> NN1 Accuracy: {res['nn1_acc']:.2f}% ({res['nn1_correct']}/{res['total']})")
        print(f"  -> NN2 Accuracy: {res['nn2_acc']:.2f}% ({res['nn2_correct']}/{res['total']})")
        
    print("=" * 60 + "\n")

else:
    BASELINE_ACC_1000 = 0.87

# Error-Generic Attacks


## Fast Gradient Sign Method (FGSM)
This step evaluates the effect of the FGSM error-generic (untargeted) attack on NN1
across a range of epsilon values. For each epsilon, adversarial examples are generated
in batches, the model accuracy is re-evaluated, and the results are saved. A Security
Evaluation Curve is plotted at the end of the sweep, and a CSV summary of all results
is written to the results directory.



In [ ]:
if RUN_FGSM_GENERIC:
    # ==============================================================================
    # Fast Gradient Sign Method (FGSM) — Error-Generic Attack
    # ==============================================================================

    print("Initializing FGSM Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME = "FGSM"
    EPSILONS    = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across {len(EPSILONS)} epsilon values...")

    results    = []
    accuracies = []

    for eps in EPSILONS:
        print(f"\n--- Evaluating Epsilon: {eps:.3f} ---")

        fgsm = FastGradientMethod(estimator=art_classifier, eps=eps)

        correct_predictions = 0
        total_processed     = 0
        start_time          = time.time()

        for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
            batch_paths    = image_paths[i:i + BATCH_SIZE]
            batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

            batch_tensors  = []
            batch_filenames = []
            for path in batch_paths:
                img = Image.open(path).convert('RGB')
                batch_tensors.append(preprocess_attack(img).numpy())
                batch_filenames.append(os.path.basename(path))

            x_batch = np.stack(batch_tensors)
            x_adv   = fgsm.generate(x=x_batch)

            preds        = art_classifier.predict(x_adv)
            pred_indices = np.argmax(preds, axis=1)

            for j, pred_idx in enumerate(pred_indices):
                raw_label = LABELS[pred_idx]
                if isinstance(raw_label, np.ndarray):
                    raw_label = raw_label.flatten()[0]
                if isinstance(raw_label, bytes):
                    raw_label = raw_label.decode('utf-8')

                predicted_name = str(raw_label).replace("b'", "").replace("'", "").replace("[", "").replace("]", "").strip()
                true_name      = id_to_name.get(batch_true_ids[j], "Unknown")

                if normalize_label(predicted_name) == normalize_label(true_name):
                    correct_predictions += 1

            save_adversarial_images(x_adv, batch_true_ids, batch_filenames, ATTACK_NAME, eps)
            total_processed += len(batch_paths)

        elapsed = time.time() - start_time
        acc     = (correct_predictions / total_processed) * 100
        accuracies.append(acc)

        print(f"Accuracy at eps={eps:.3f}: {acc:.2f}%")

        results.append({
            'attack_name':          ATTACK_NAME,
            'epsilon':              eps,
            'correct_predictions':  correct_predictions,
            'total_images':         total_processed,
            'accuracy':             round(acc, 2),
            'time_elapsed_s':       round(elapsed, 2)
        })

    # ------------------------------------------------------------------------------
    # 3. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results  = pd.DataFrame(results)
    csv_path    = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 4. Security Evaluation Curve
    # ------------------------------------------------------------------------------
    plot_security_evaluation_curve(EPSILONS, accuracies, ATTACK_NAME, ATTACK_RESULTS_DIR, BASELINE_ACC_1000)

    # ------------------------------------------------------------------------------
    # 5. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:  {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Baseline Accuracy:        {BASELINE_ACC:.2f}%")
    print(f"Lowest Accuracy:          {accuracies[-1]:.2f}%")
    print("-" * 50)
    print(f"Total Model Degradation:  {(BASELINE_ACC - accuracies[-1]):.2f}%")
    print("=" * 50 + "\n")

## Basic Iterative Method (BIM) — Error-Generic Attack

This step evaluates the effect of the BIM error-generic (untargeted) attack on NN1.
BIM extends FGSM by applying multiple smaller gradient steps, each of size
`eps_step = eps / max_iter`, staying within the L-infinity ball of radius epsilon.
The attack is evaluated across a range of epsilon values for each value of `max_iter`,
producing one Security Evaluation Curve per configuration, a comparative SEC overlaying
all curves, and a hyperparameter effect plot showing accuracy as a function of `max_iter`
at fixed epsilon values. A CSV summary of all results is written to the results directory.

In [ ]:
if RUN_BIM_GENERIC:
    # ==============================================================================
    # Basic Iterative Method (BIM) — Error-Generic Attack
    # ==============================================================================

    print("Initializing BIM Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME   = "BIM"
    EPSILONS      = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST = [1, 5, 10, 20]

    # One distinct color per max_iter value — reused across all plots for consistency
    ITER_COLORS = {
        1:  '#1f77b4',
        5:  "#ffbf00",
        10: '#2ca02c',
        20: '#d62728'
    }

    # Fixed epsilon values used in the hyperparameter effect plot (low, medium, high budget)
    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across " f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values...")

    results        = []
    all_accuracies = {}  # max_iter -> list of accuracies, one per epsilon

    for max_iter in MAX_ITER_LIST:
        print(f"\n{'='*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'='*50}")

        accuracies = []

        for eps in EPSILONS:
            print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

            eps_step = eps / max_iter

            bim = BasicIterativeMethod(
                estimator=art_classifier,
                eps=eps,
                eps_step=eps_step,
                max_iter=max_iter
            )

            correct_predictions = 0
            total_processed     = 0
            start_time          = time.time()

            for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                batch_paths     = image_paths[i:i + BATCH_SIZE]
                batch_true_ids  = true_folder_ids[i:i + BATCH_SIZE]

                batch_tensors   = []
                batch_filenames = []
                for path in batch_paths:
                    img = Image.open(path).convert('RGB')
                    batch_tensors.append(preprocess_attack(img).numpy())
                    batch_filenames.append(os.path.basename(path))

                x_batch = np.stack(batch_tensors)
                x_adv   = bim.generate(x=x_batch)

                preds        = art_classifier.predict(x_adv)
                pred_indices = np.argmax(preds, axis=1)

                for j, pred_idx in enumerate(pred_indices):
                    raw_label = LABELS[pred_idx]
                    if isinstance(raw_label, np.ndarray):
                        raw_label = raw_label.flatten()[0]
                    if isinstance(raw_label, bytes):
                        raw_label = raw_label.decode('utf-8')

                    predicted_name = (str(raw_label)
                                    .replace("b'", "").replace("'", "")
                                    .replace("[", "").replace("]", "").strip())
                    true_name = id_to_name.get(batch_true_ids[j], "Unknown")

                    if normalize_label(predicted_name) == normalize_label(true_name):
                        correct_predictions += 1

                save_adversarial_images(
                    x_adv, batch_true_ids, batch_filenames,
                    f"{ATTACK_NAME}_iter{max_iter}", eps
                )
                total_processed += len(batch_paths)

            elapsed = time.time() - start_time
            acc     = (correct_predictions / total_processed) * 100
            accuracies.append(acc)

            print(f"Accuracy at eps={eps:.3f} | max_iter={max_iter}: {acc:.2f}%")

            results.append({
                'attack_name':         ATTACK_NAME,
                'max_iter':            max_iter,
                'eps_step':            round(eps / max_iter, 6),
                'epsilon':             eps,
                'correct_predictions': correct_predictions,
                'total_images':        total_processed,
                'accuracy':            round(acc, 2),
                'time_elapsed_s':      round(elapsed, 2)
            })

        all_accuracies[max_iter] = accuracies

        # --------------------------------------------------------------------------
        # 3. Individual SEC for this max_iter value
        # --------------------------------------------------------------------------
        color = ITER_COLORS[max_iter]

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                linestyle='-', color=color, linewidth=1.5)
        ax.plot(EPSILONS, accuracies,
                marker='o', linestyle='-', color=color,
                linewidth=1.5, markersize=6,
                label=f'NN1 Performance ({ATTACK_NAME}, max_iter={max_iter})')
        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor=color,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} (max_iter={max_iter})',fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_iter{max_iter}_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Individual SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # 4. Comparative SEC — all max_iter curves overlaid, epsilon on x-axis
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC,
            marker='o', markerfacecolor='black',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    for max_iter, accuracies in all_accuracies.items():
        color = ITER_COLORS[max_iter]
        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                linestyle='-', color=color, linewidth=1.5)
        ax.plot(EPSILONS, accuracies,
                marker='o', linestyle='-', color=color,
                linewidth=1.5, markersize=6,
                label=f'{ATTACK_NAME} max_iter={max_iter}')

    ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} (Comparative)',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    comparative_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_comparative_SEC.png')
    plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

    # ------------------------------------------------------------------------------
    # 5. Hyperparameter Effect Plot — Accuracy vs max_iter at fixed epsilon values
    #    eps_step = eps / max_iter for each point, all other parameters fixed.
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[m][eps_idx] for m in MAX_ITER_LIST]

        ax.plot(MAX_ITER_LIST, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--', linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of max_iter on Model Accuracy — {ATTACK_NAME}',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(MAX_ITER_LIST)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_plot_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_maxiter_effect.png')
    plt.savefig(effect_plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Hyperparameter effect plot saved at: {effect_plot_path}")

    # ------------------------------------------------------------------------------
    # 6. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 7. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:  {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Max Iter Values:          {MAX_ITER_LIST}")
    print(f"Baseline Accuracy:        {BASELINE_ACC:.2f}%")
    for max_iter, accuracies in all_accuracies.items():
        print(f"  max_iter={max_iter:2d} — Lowest Accuracy: {accuracies[-1]:.2f}%  " f"(Degradation: {BASELINE_ACC - accuracies[-1]:.2f}%)")
    print("=" * 50 + "\n")

## Projected Gradient Descent (PGD) — Error-Generic Attack

This step evaluates the effect of the PGD error-generic (untargeted) attack on NN1.
PGD extends BIM by adding random restarts: before each optimization, the starting
point is perturbed by a random offset sampled uniformly within the L-infinity ball
of radius epsilon. The parameter `num_random_init` controls how many random restarts
are performed, with `num_random_init=0` reducing PGD to standard BIM.
The attack is evaluated across all combinations of `epsilon`, `max_iter`, and
`num_random_init`, producing individual SECs, comparative plots, and hyperparameter
effect plots. A CSV summary of all results is written to the results directory.

In [ ]:
if RUN_PGD_GENERIC:
    # ==============================================================================
    # STEP 3 Projected Gradient Descent (PGD) — Error-Generic Attack
    # ==============================================================================

    print("Initializing PGD Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME        = "PGD"
    BATCH_SIZE         = 128
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]

    # One color per max_iter — reused across all SEC plots
    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ffbf00',
        10: '#2ca02c',
        20: '#d62728',
        40: '#9467bd'
    }

    # One color per num_random_init — reused across all random_init plots
    RAND_COLORS = {
        0: '#1f77b4',
        1: '#ff7f0e',
        3: '#2ca02c',
        5: '#d62728'
    }

    # Fixed epsilon values for hyperparameter effect plots (low, medium, high budget)
    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
    os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # all_accuracies[num_random_init][max_iter] -> list of accuracies, one per epsilon
    # ------------------------------------------------------------------------------
    print(f"\nStarting {ATTACK_NAME} generation across "
            f"{len(NUM_RANDOM_INITS)} num_random_init values x "
            f"{len(MAX_ITER_LIST)} max_iter values x "
            f"{len(EPSILONS)} epsilon values...")

    results        = []
    all_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}

    for num_random_init in NUM_RANDOM_INITS:
        print(f"\n{'#'*50}")
        print(f"NUM_RANDOM_INIT = {num_random_init}")
        print(f"{'#'*50}")

        for max_iter in MAX_ITER_LIST:
            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | NUM_RANDOM_INIT = {num_random_init}")
            print(f"{'='*50}")

            accuracies = []

            for eps in EPSILONS:
                print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} | num_random_init: {num_random_init} ---")

                eps_step = eps / max_iter

                pgd = ProjectedGradientDescentPyTorch(
                    estimator=art_classifier,
                    norm=np.inf,
                    eps=eps,
                    eps_step=eps_step,
                    max_iter=max_iter,
                    num_random_init=num_random_init,
                    targeted=False,
                    batch_size=BATCH_SIZE,
                    verbose=False
                )

                correct_predictions = 0
                total_processed     = 0
                start_time          = time.time()

                for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                    batch_paths     = image_paths[i:i + BATCH_SIZE]
                    batch_true_ids  = true_folder_ids[i:i + BATCH_SIZE]

                    batch_tensors   = []
                    batch_filenames = []
                    for path in batch_paths:
                        img = Image.open(path).convert('RGB')
                        batch_tensors.append(preprocess_attack(img).numpy())
                        batch_filenames.append(os.path.basename(path))

                    x_batch = np.stack(batch_tensors)
                    x_adv   = pgd.generate(x=x_batch)

                    preds        = art_classifier.predict(x_adv)
                    pred_indices = np.argmax(preds, axis=1)

                    for j, pred_idx in enumerate(pred_indices):
                        raw_label = LABELS[pred_idx]
                        if isinstance(raw_label, np.ndarray):
                            raw_label = raw_label.flatten()[0]
                        if isinstance(raw_label, bytes):
                            raw_label = raw_label.decode('utf-8')

                        predicted_name = (str(raw_label)
                                        .replace("b'", "").replace("'", "")
                                        .replace("[", "").replace("]", "").strip())
                        true_name = id_to_name.get(batch_true_ids[j], "Unknown")

                        if normalize_label(predicted_name) == normalize_label(true_name):
                            correct_predictions += 1

                    save_adversarial_images(
                        x_adv, batch_true_ids, batch_filenames,
                        f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}", eps
                    )
                    total_processed += len(batch_paths)

                elapsed = time.time() - start_time
                acc     = (correct_predictions / total_processed) * 100
                accuracies.append(acc)

                print(f"Accuracy at eps={eps:.3f} | max_iter={max_iter} | num_random_init={num_random_init}: {acc:.2f}%")

                results.append({
                    'attack_name':         ATTACK_NAME,
                    'num_random_init':     num_random_init,
                    'max_iter':            max_iter,
                    'eps_step':            round(eps / max_iter, 6),
                    'epsilon':             eps,
                    'correct_predictions': correct_predictions,
                    'total_images':        total_processed,
                    'accuracy':            round(acc, 2),
                    'time_elapsed_s':      round(elapsed, 2)
                })

            all_accuracies[num_random_init][max_iter] = accuracies

            # ----------------------------------------------------------------------
            # 3. Individual SEC for this (num_random_init, max_iter) combination
            # ----------------------------------------------------------------------
            color = ITER_COLORS[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 Performance ({ATTACK_NAME}, max_iter={max_iter}, nri={num_random_init})')
            ax.plot(0.0, BASELINE_ACC,
                    marker='o', markerfacecolor=color,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

            ax.set_title(f'Security Evaluation Curve — {ATTACK_NAME} 'f'(max_iter={max_iter}, num_random_init={num_random_init})',fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Model Accuracy (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper right', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Individual SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # 4. Comparative SEC per num_random_init — max_iter varies, epsilon on x-axis
    #    For each fixed num_random_init, overlay all max_iter curves.
    #    Ceteris paribus: num_random_init fixed, max_iter varies.
    # ------------------------------------------------------------------------------
    for num_random_init in NUM_RANDOM_INITS:
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        for max_iter in MAX_ITER_LIST:
            color      = ITER_COLORS[max_iter]
            accuracies = all_accuracies[num_random_init][max_iter]
            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.set_title(f'SEC — {ATTACK_NAME} | num_random_init={num_random_init} (max_iter varies)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        path = os.path.join(ATTACK_RESULTS_DIR,
                            f'{ATTACK_NAME}_nri{num_random_init}_comparative_maxiter_SEC.png')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC (nri={num_random_init}, max_iter varies) saved at: {path}")

    # ------------------------------------------------------------------------------
    # 5. Comparative SEC per max_iter — num_random_init varies, epsilon on x-axis
    #    For each fixed max_iter, overlay all num_random_init curves.
    #    Ceteris paribus: max_iter fixed, num_random_init varies.
    # ------------------------------------------------------------------------------
    for max_iter in MAX_ITER_LIST:
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC,
                marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

        for num_random_init in NUM_RANDOM_INITS:
            color      = RAND_COLORS[num_random_init]
            accuracies = all_accuracies[num_random_init][max_iter]
            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC, accuracies[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, accuracies,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'num_random_init={num_random_init}')

        ax.set_title(f'SEC — {ATTACK_NAME} | max_iter={max_iter} (num_random_init varies)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        path = os.path.join(ATTACK_RESULTS_DIR,
                            f'{ATTACK_NAME}_iter{max_iter}_comparative_nri_SEC.png')
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC (max_iter={max_iter}, nri varies) saved at: {path}")

    # ------------------------------------------------------------------------------
    # 6. Hyperparameter Effect Plot — Accuracy vs max_iter at fixed epsilon values
    #    num_random_init fixed at maximum value (5) — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_NRI_FOR_EFFECT = max(NUM_RANDOM_INITS)

    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[FIXED_NRI_FOR_EFFECT][m][eps_idx] for m in MAX_ITER_LIST]

        ax.plot(MAX_ITER_LIST, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--',linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of max_iter on Model Accuracy — {ATTACK_NAME} 'f'(num_random_init={FIXED_NRI_FOR_EFFECT})',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(MAX_ITER_LIST)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_iter_path = os.path.join(ATTACK_RESULTS_DIR,
                                    f'{ATTACK_NAME}_maxiter_effect_nri{FIXED_NRI_FOR_EFFECT}.png')
    plt.savefig(effect_iter_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] max_iter effect plot saved at: {effect_iter_path}")

    # ------------------------------------------------------------------------------
    # 7. Hyperparameter Effect Plot — Accuracy vs num_random_init at fixed epsilon values
    #    max_iter fixed at maximum value (40) — ceteris paribus.
    # ------------------------------------------------------------------------------
    FIXED_ITER_FOR_EFFECT = max(MAX_ITER_LIST)

    fig, ax = plt.subplots(figsize=(9, 6))

    for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
        eps_idx     = EPSILONS.index(fixed_eps)
        accs_at_eps = [all_accuracies[nri][FIXED_ITER_FOR_EFFECT][eps_idx] for nri in NUM_RANDOM_INITS]

        ax.plot(NUM_RANDOM_INITS, accs_at_eps,
                marker='o', linestyle='-',
                color=EPSILON_EFFECT_COLORS[fixed_eps],
                linewidth=1.5, markersize=6,
                label=rf'$\epsilon$ = {fixed_eps}')

    ax.axhline(y=BASELINE_ACC, color='#2ca02c', linestyle='--',linewidth=1.5, label=f'Baseline Accuracy ({BASELINE_ACC:.2f}%)')

    ax.set_title(f'Effect of num_random_init on Model Accuracy — {ATTACK_NAME} 'f'(max_iter={FIXED_ITER_FOR_EFFECT})',fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Number of Random Restarts (num_random_init)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.set_xticks(NUM_RANDOM_INITS)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    effect_nri_path = os.path.join(ATTACK_RESULTS_DIR,f'{ATTACK_NAME}_nri_effect_iter{FIXED_ITER_FOR_EFFECT}.png')
    plt.savefig(effect_nri_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] num_random_init effect plot saved at: {effect_nri_path}")

    # ------------------------------------------------------------------------------
    # 8. Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # 9. Attack Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 50)
    print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
    print("=" * 50)
    print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
    print(f"Max Iter Values:           {MAX_ITER_LIST}")
    print(f"Num Random Init Values:    {NUM_RANDOM_INITS}")
    print(f"Baseline Accuracy:         {BASELINE_ACC:.2f}%")
    print("-" * 50)
    for num_random_init in NUM_RANDOM_INITS:
        for max_iter in MAX_ITER_LIST:
            acc_worst = all_accuracies[num_random_init][max_iter][-1]
            print(f"  nri={num_random_init} | max_iter={max_iter:2d} — "
                f"Lowest Accuracy: {acc_worst:.2f}%  "
                f"(Degradation: {BASELINE_ACC - acc_worst:.2f}%)")
    print("=" * 50 + "\n")


In [ ]:
if RUN_CW_GENERIC:
    # ==============================================================================
    # Carlini & Wagner (CW) — Error-Generic Attack
    # ==============================================================================

    os.makedirs(RESULTS_DIR, exist_ok=True)

    # The visual range is [0.0, 1.0]. The maximum L_inf constraint is 0.1.
    EPSILON_LIMIT = 0.1

    print("Reading of CSV nad sync of IDs...")
    df_subjects = pd.read_csv(CSV_PATH)
    df_subjects['Class_ID'] = df_subjects['Class_ID'].astype(str).str.strip()
    df_subjects['Name'] = df_subjects['Name'].astype(str).str.strip()

    label_to_index = {}
    for _, row in df_subjects.iterrows():
        c_id = row['Class_ID']
        p_name = row['Name']
        found_idx = -1
        for idx, lbl in enumerate(LABELS):
            lbl_str = str(lbl).strip().strip("'").strip('"')
            if c_id in lbl_str or p_name in lbl_str:
                found_idx = idx
                break
        if found_idx != -1:
            label_to_index[c_id] = found_idx
        else:
            print(f"WARNING: Can't find: '{c_id}' or '{p_name}'")

    print(f"Mapping completed with success. Find {len(label_to_index)} subjects.")

    print(f"--- START SCRIPT - WORKER {WORKER} on {device} ---")
    print("\n[Phase 1] Preprocessing and loading images...")

    preprocess_attack = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ])

    x_data = []
    y_labels_numeric = []
    image_names = []

    image_files = sorted([f for f in os.listdir(DATA_DIR_100) if f.lower().endswith(('.jpg', '.png'))])

    if len(image_files) != 100:
        print(f"WARNING: Found {len(image_files)} images instead of 100!")

    for img_name in image_files:
        img_path = os.path.join(DATA_DIR_100, img_name)

        img = Image.open(img_path).convert('RGB')
        img_tensor = preprocess_attack(img)
        x_data.append(img_tensor.numpy())
        image_names.append(img_name)

        class_id = os.path.splitext(img_name)[0].strip()

        if class_id in label_to_index:
            y_labels_numeric.append(label_to_index[class_id])
        else:
            print(f"Critical error: ID {class_id} not mapped.")
            y_labels_numeric.append(0)

    x_data = np.array(x_data)
    num_classes = len(LABELS)
    y_data_one_hot = np.eye(num_classes)[y_labels_numeric]

    print(f"Loaded {len(x_data)} images. Shape: {x_data.shape}")

    # ==============================================================================
    # WRAPPER ART AND BASELINE EVALUATION
    # ==============================================================================
    print("\n[Phase 3] Configuring ART Classifier...")

    loss_fn = torch.nn.CrossEntropyLoss()

    classifier = PyTorchClassifier(
        model=nn1,
        clip_values=(0.0, 1.0),
        loss=loss_fn,
        optimizer=None,
        input_shape=(3, 160, 160),
        nb_classes=num_classes,
        preprocessing=(np.array([0.5, 0.5, 0.5]), np.array([0.5, 0.5, 0.5])),
        device_type='gpu' if torch.cuda.is_available() else 'cpu'
    )

    if WORKER == "Christian":
        max_iter_list = [1]
        start_idx, end_idx = 0, 100
    elif WORKER == "Salvatore":
        max_iter_list = [5]
        start_idx, end_idx = 0, 100
    elif WORKER == "Candido":
        max_iter_list = [10]
        start_idx, end_idx = 0, 100
    else:
        raise ValueError("Invalid WORKER. Select one from this list: [Christian, Candido, Salvatore]")

    x_test_subset = x_data[start_idx:end_idx]
    y_test_subset = y_data_one_hot[start_idx:end_idx]
    image_names_subset = image_names[start_idx:end_idx]

    print(f"Worker {WORKER}: assigned {len(x_test_subset)} images.")

    # --- CLEAN BASELINE COMPUTATION ---
    print("\nComputing baseline on original images...")
    preds_clean = np.argmax(classifier.predict(x_test_subset), axis=1)
    true_labels = np.argmax(y_test_subset, axis=1)
    correct_clean_mask = (preds_clean == true_labels)
    num_correct_clean = np.sum(correct_clean_mask)
    baseline_acc = (num_correct_clean / len(x_test_subset)) * 100

    print(f"Baseline Accuracy (Clean): {num_correct_clean}/{len(x_test_subset)} ({baseline_acc:.2f}%) correctly classified.")

    # ==============================================================================
    # GRID SEARCH CW L_INF (FILTER OUT L_INF > 0.10)
    # ==============================================================================
    print("\n[Phase 4] Starting C&W L_inf attack (skipping perturbations > 0.10)...")

    learning_rate_list = [0.001, 0.01, 0.05, 0.1]
    confidence_list = [0.0, 0.5, 1.0]

    results = []
    total_combinations = len(max_iter_list) * len(learning_rate_list) * len(confidence_list)
    current_combo = 1

    for m_iter in max_iter_list:
        for lr in learning_rate_list:
            for conf in confidence_list:
                print(f"\n[{current_combo}/{total_combinations}] Configuration: max_iter={m_iter}, lr={lr}, confidence={conf}")
                start_time_combo = time.time()

                # Build the output path string
                lr_str = str(lr).replace('.', '_')
                conf_str = str(conf).replace('.', '_')
                combo_dir_name = f"max_iter_{m_iter}_lr_{lr_str}_c_{conf_str}"

                output_images_dir = os.path.join(BASE_ADVERSARIAL_CW_DIR, combo_dir_name)
                os.makedirs(output_images_dir, exist_ok=True)

                attack_cw = CarliniLInfMethod(
                    classifier=classifier,
                    targeted=False,
                    max_iter=m_iter,
                    learning_rate=lr,
                    confidence=conf,
                    verbose=False
                )

                x_adv_raw_list = []
                print(f"   -> Generating for {len(x_test_subset)} images...")

                # --- Generation loop with timing ---
                for i in range(len(x_test_subset)):
                    start_img_time = time.time()

                    single_adv = attack_cw.generate(x=x_test_subset[i:i+1])

                    img_time = time.time() - start_img_time
                    print(f"      - Image {i+1:02d}/{len(x_test_subset)} ({image_names_subset[i]}) processed in {img_time:.2f} seconds")

                    x_adv_raw_list.append(single_adv)

                x_adv_raw = np.concatenate(x_adv_raw_list, axis=0)

                # Compute the raw perturbation without forced clipping
                perturbation_raw = x_adv_raw - x_test_subset

                l_inf_applied_list = []
                saved_count = 0

                # --- CHECK AND SAVE INDIVIDUAL IMAGES ---
                for i in range(len(x_adv_raw)):
                    # Compute the L_inf perturbation for the i-th image
                    l_inf_img = np.max(np.abs(perturbation_raw[i]))

                    # Add 1e-5 to avoid float-precision issues (e.g. 0.1000001) discarding valid images
                    if l_inf_img > EPSILON_LIMIT + 1e-5:
                        print(f"      - Image {image_names_subset[i]}: for this configuration the CW algorithm only found a perturbation larger than 0.10 (L_inf = {l_inf_img:.4f}) that fooled the network, so it was not saved in the current config folder.")
                        continue

                    # If it passes the check, clip pixels to the visual range [0.0, 1.0] for saving
                    adv_img_np = np.clip(x_adv_raw[i], 0.0, 1.0)

                    # Conversion for PIL saving
                    adv_img_np_save = np.transpose(adv_img_np, (1, 2, 0))
                    adv_img_np_save = (adv_img_np_save * 255.0).astype(np.uint8)
                    adv_pillow_img = Image.fromarray(adv_img_np_save)

                    # Save
                    img_save_path = os.path.join(output_images_dir, image_names_subset[i])
                    adv_pillow_img.save(img_save_path)

                    saved_count += 1

                    # Compute the actual perturbation applied to the photo (after visual clipping to [0, 1])
                    actual_perturbation = adv_img_np - x_test_subset[i]
                    l_inf_applied_list.append(np.max(np.abs(actual_perturbation)))

                # Mean perturbation for this configuration (only on valid saved images)
                if saved_count > 0:
                    mean_linf_applied = np.mean(l_inf_applied_list)
                else:
                    mean_linf_applied = 0.0

                elapsed_time_combo = time.time() - start_time_combo

                # CSV log entry
                results.append({
                    'WORKER': WORKER,
                    'max_iter': m_iter,
                    'learning_rate': lr,
                    'confidence': conf,
                    'Mean_Linf_Applied': round(mean_linf_applied, 4),
                    'Time_Elapsed (s)': round(elapsed_time_combo, 2)
                })

                print(f" --> RESULT: Saved {saved_count}/{len(x_test_subset)} images | Mean L_inf applied: {mean_linf_applied:.4f} | Total Config Time: {elapsed_time_combo:.1f}s")
                current_combo += 1

    # ==============================================================================
    # PHASE 5: SAVE WORKER DATA
    # ==============================================================================
    df_results = pd.DataFrame(results)
    csv_filename = f'cw_grid_search_worker_{WORKER}.csv'
    csv_filepath = os.path.join(RESULTS_DIR, csv_filename)

    df_results.to_csv(csv_filepath, index=False)

    print("\n=======================================================")
    print(f"Work completed! Results saved at:\n{csv_filepath}")
    print("=======================================================")

### Creation of various plots for CW untargeted attack

In [ ]:
RUN_EVALUATION_CW = False

if RUN_EVALUATION_CW:
    # ==========================================
    # 1. PATHS AND CONFIGURATION
    # ==========================================
    # Paths to your adversarial directories and output files
    BASE_DIR = r"./adversarial_data/CW_Untargeted"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # ==========================================
    # 2. LABEL MAPPING (Extracting Ground Truth)
    # ==========================================
    print("Loading VGGFace2 labels and mapping subjects...")
    try:
        df_subjects = pd.read_csv(CSV_PATH)
        df_subjects['Class_ID'] = df_subjects['Class_ID'].astype(str).str.strip()
        df_subjects['Name'] = df_subjects['Name'].astype(str).str.strip()

        label_to_index = {}
        for _, row in df_subjects.iterrows():
            c_id = row['Class_ID']
            p_name = row['Name']
            found_idx = -1

            for idx, lbl in enumerate(LABELS):
                lbl_str = str(lbl).strip().strip("'").strip('"')
                if c_id in lbl_str or p_name in lbl_str:
                    found_idx = idx
                    break

            if found_idx != -1:
                label_to_index[c_id] = found_idx
            else:
                print(f"[WARNING] Can't find mapping for: '{c_id}' or '{p_name}'")

        print(f"Mapping completed successfully. Found {len(label_to_index)} subjects.")
    except Exception as e:
        print(f"[ERROR] Failed to load label mapping: {e}")
        print("Please check CSV_SUBJECTS_PATH and NPY_LABELS_PATH.")
        exit(1)

    # ==========================================
    # 3. LOADING MODEL AND PRE-PROCESSING
    # ==========================================
    print("Loading Target model NN1 (InceptionResnetV1)...")
    model = InceptionResnetV1(pretrained='vggface2', classify=True).eval().to(device)

    # CRITICAL FIX: Added Normalize to match ART preprocessing (0.5 mean, 0.5 std)
    preprocess = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    # ==========================================
    # 4. LOAD AND MERGE WORKER CSVs
    # ==========================================
    print("Merging worker configurations...")
    df_candido = pd.read_csv(CSV_CANDIDO)
    df_christian = pd.read_csv(CSV_CHRISTIAN)

    df_grid = pd.concat([df_candido, df_christian], ignore_index=True)
    df_grid = df_grid.sort_values(by=['max_iter', 'learning_rate', 'confidence']).reset_index(drop=True)

    print(f"Total combinations loaded from CSVs: {len(df_grid)}")

    # ==========================================
    # 5. EVALUATION LOOP
    # ==========================================
    accuracies = []
    processed_counts = []

    with tqdm(total=len(df_grid), desc="Computing Accuracy") as pbar:
        for index, row in df_grid.iterrows():

            # Extract parameters
            max_iter = int(row['max_iter'])
            lr = row['learning_rate']
            conf = row['confidence']

            # Format folder name based on parameters
            lr_str = str(lr).replace('.', '_')
            conf_str = str(conf).replace('.', '_')

            name_dir = f"max_iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
            path_dir = os.path.join(BASE_DIR, name_dir)

            # Handle missing directories (e.g., if a configuration completely failed to generate)
            if not os.path.exists(path_dir):
                accuracies.append(0.0)
                processed_counts.append(0)
                pbar.update(1)
                continue

            current_count = 0
            total_count = 0

            # Iterate over all adversarial images generated for this configuration
            for file in os.listdir(path_dir):
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(path_dir, file)

                    # Extract Class_ID from filename (e.g., 'n000001.jpg' -> 'n000001')
                    class_id = os.path.splitext(file)[0].strip()

                    # Get ground truth label. Skip if not found in our mapping.
                    if class_id not in label_to_index:
                        print(f"\n[Warning] ID {class_id} not mapped. Skipping image.")
                        continue

                    y_true = label_to_index[class_id]

                    try:
                        # Load and preprocess image
                        img = Image.open(img_path).convert('RGB')
                        img_tensor = preprocess(img).unsqueeze(0).to(device)

                        # Model inference
                        with torch.no_grad():
                            output_logits = model(img_tensor)
                            y_pred = torch.argmax(output_logits, dim=1).item()

                        # Check if prediction matches ground truth
                        if y_pred == y_true:
                            current_count += 1

                        total_count += 1

                    except Exception as e:
                        print(f"\n[Error] Failed processing {file}: {e}")
                        continue

            # Calculate accuracy for the current folder
            accuracy = (current_count / total_count) if total_count > 0 else 0.0

            accuracies.append(accuracy)
            processed_counts.append(total_count)

            pbar.update(1)

    # ==========================================
    # 6. SAVE FINAL RESULTS
    # ==========================================
    df_grid['accuracy'] = accuracies
    df_grid['img_evaluated'] = processed_counts

    df_grid.to_csv(OUTPUT_CSV_CW, index=False)
    print(f"\n[Phase 1 Completed] Final results saved to: {OUTPUT_CSV_CW}")

## Fast Gradient Sign Method (FGSM) — Error-Specific (Targeted) Attack

This step evaluates the FGSM attack in its targeted (error-specific) variant on NN1.
Two targeting strategies are evaluated:
- **Fixed**: every image is pushed toward a single fixed target class (impersonation attack).
- **LeastLikely**: each image is pushed toward the class assigned the lowest probability
  by the clean model, maximizing the semantic distance of the misclassification.

For targeted attacks the metric reported is the Attack Success Rate (ASR), defined as
the fraction of images for which the model predicts exactly the intended target class.
A CSV summary and a Targeted Security Evaluation Curve are saved for each mode.


In [ ]:
if RUN_FGSM_SPECIFIC:
    # ==============================================================================
    # STEP Fast Gradient Sign Method (FGSM) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing FGSM Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "FGSM_Targeted"
    BATCH_SIZE         = 32
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026  
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

        print(f"\n{'='*60}")
        print(f"Starting {ATTACK_NAME} across {len(EPSILONS)} epsilon values")
        print(f"{'='*60}")

        results              = []
        attack_success_rates = []

        for eps in EPSILONS:
            print(f"\n--- Evaluating Epsilon: {eps:.3f} | Mode: {mode} ---")

            fgsm = FastGradientMethod(estimator=art_classifier, eps=eps, targeted=True)

            successful_attacks = 0
            total_processed    = 0
            start_time         = time.time()

            for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                batch_paths    = image_paths[i:i + BATCH_SIZE]
                batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                batch_tensors   = []
                batch_filenames = []
                for path in batch_paths:
                    img = Image.open(path).convert('RGB')
                    batch_tensors.append(preprocess_attack(img).numpy())
                    batch_filenames.append(os.path.basename(path))

                x_batch = np.stack(batch_tensors)

                # Determine target indices — ceteris paribus: only targeting strategy varies
                if mode == "LeastLikely":
                    batch_target_indices = get_least_likely_indices(x_batch)
                elif mode == "Fixed":
                    batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                # Build one-hot encoded target array required by ART targeted attacks
                y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                for k, t_idx in enumerate(batch_target_indices):
                    y_target_one_hot[k, int(t_idx)] = 1.0

                x_adv        = fgsm.generate(x=x_batch, y=y_target_one_hot)
                preds        = art_classifier.predict(x_adv)
                pred_indices = np.argmax(preds, axis=1)

                for j, pred_idx in enumerate(pred_indices):
                    if pred_idx == batch_target_indices[j]:
                        successful_attacks += 1

                save_adversarial_images(
                    x_adv, batch_true_ids, batch_filenames, ATTACK_NAME, eps
                )
                total_processed += len(batch_paths)

            elapsed        = time.time() - start_time
            asr_percentage = (successful_attacks / total_processed) * 100
            attack_success_rates.append(asr_percentage)

            print(f"ASR at eps={eps:.3f}: {asr_percentage:.2f}%")

            results.append({
                'attack_name':         ATTACK_NAME,
                'target_mode':         mode,
                'epsilon':             eps,
                'successful_attacks':  successful_attacks,
                'total_images':        total_processed,
                'attack_success_rate': round(asr_percentage, 2),
                'time_elapsed_s':      round(elapsed, 2)
            })

        # --------------------------------------------------------------------------
        # 3. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 4. Targeted Security Evaluation Curve
        # --------------------------------------------------------------------------
        plot_targeted_sec(EPSILONS, attack_success_rates, ATTACK_NAME, ATTACK_RESULTS_DIR)

        # --------------------------------------------------------------------------
        # 5. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:             {mode}")
        print(f"Evaluated Epsilon Range: {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Highest ASR:             {attack_success_rates[-1]:.2f}% at eps={EPSILONS[-1]}")
        print("=" * 50 + "\n")

## Basic Iterative Method (BIM) — Error-Specific (Targeted) Attack

This step evaluates BIM in its targeted (error-specific) variant on NN1 across two
targeting strategies (Fixed and LeastLikely) and a range of `max_iter` values.
The step size is set to `eps / max_iter` for each configuration, keeping all other
parameters fixed (ceteris paribus). The Attack Success Rate (ASR) is reported for
each combination. Individual and comparative Targeted SECs and a hyperparameter
effect plot are saved alongside a CSV summary.

In [ ]:
if RUN_BIM_SPECIFIC: 
    # ==============================================================================
    # Basic Iterative Method (BIM) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing BIM Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "BIM_Targeted"
    BATCH_SIZE         = 32
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ff7f0e',
        10: '#2ca02c',
        20: '#d62728'
    }

    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} across "
            f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values")
        print(f"{'*'*70}")

        results  = []
        all_asrs = {}  # max_iter -> list of ASRs, one per epsilon

        for max_iter in MAX_ITER_LIST:
            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | MODE = {mode}")
            print(f"{'='*50}")

            attack_success_rates = []

            for eps in EPSILONS:
                print(f"\n--- Evaluating Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

                eps_step = eps / max_iter

                bim = BasicIterativeMethod(
                    estimator=art_classifier,
                    eps=eps,
                    eps_step=eps_step,
                    max_iter=max_iter,
                    targeted=True,
                    batch_size=BATCH_SIZE
                )

                successful_attacks = 0
                total_processed    = 0
                start_time         = time.time()

                for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                    batch_paths    = image_paths[i:i + BATCH_SIZE]
                    batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                    batch_tensors   = []
                    batch_filenames = []
                    for path in batch_paths:
                        img = Image.open(path).convert('RGB')
                        batch_tensors.append(preprocess_attack(img).numpy())
                        batch_filenames.append(os.path.basename(path))

                    x_batch = np.stack(batch_tensors)

                    if mode == "LeastLikely":
                        batch_target_indices = get_least_likely_indices(x_batch)
                    elif mode == "Fixed":
                        batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                    y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                    for k, t_idx in enumerate(batch_target_indices):
                        y_target_one_hot[k, int(t_idx)] = 1.0

                    x_adv        = bim.generate(x=x_batch, y=y_target_one_hot)
                    preds        = art_classifier.predict(x_adv)
                    pred_indices = np.argmax(preds, axis=1)

                    for j, pred_idx in enumerate(pred_indices):
                        if pred_idx == batch_target_indices[j]:
                            successful_attacks += 1

                    save_adversarial_images(
                        x_adv, batch_true_ids, batch_filenames,
                        f"{ATTACK_NAME}_iter{max_iter}", eps
                    )
                    total_processed += len(batch_paths)

                elapsed = time.time() - start_time
                asr     = (successful_attacks / total_processed) * 100
                attack_success_rates.append(asr)

                print(f"ASR at eps={eps:.3f} | max_iter={max_iter}: {asr:.2f}%")

                results.append({
                    'attack_name':         ATTACK_NAME,
                    'target_mode':         mode,
                    'max_iter':            max_iter,
                    'eps_step':            round(eps_step, 6),
                    'epsilon':             eps,
                    'successful_attacks':  successful_attacks,
                    'total_images':        total_processed,
                    'attack_success_rate': round(asr, 2),
                    'time_elapsed_s':      round(elapsed, 2)
                })

            all_asrs[max_iter] = attack_success_rates

            # ----------------------------------------------------------------------
            # 3. Individual Targeted SEC for this max_iter value
            # ----------------------------------------------------------------------
            color = ITER_COLORS[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot([0.0, EPSILONS[0]], [0.0, attack_success_rates[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, attack_success_rates,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'ASR ({ATTACK_NAME}, max_iter={max_iter})')
            ax.plot(0.0, 0.0,
                    marker='o', markerfacecolor=color,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} (max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(ATTACK_RESULTS_DIR,
                                    f'{ATTACK_NAME}_iter{max_iter}_SEC.png')
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Individual SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # 4. Comparative Targeted SEC — all max_iter curves overlaid
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        for max_iter, asrs in all_asrs.items():
            color = ITER_COLORS[max_iter]
            ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                    linestyle='-', color=color, linewidth=1.5)
            ax.plot(EPSILONS, asrs,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6, label=f'max_iter={max_iter}')

        ax.set_title(f'Targeted SEC — {ATTACK_NAME} (Comparative)',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        comparative_path = os.path.join(ATTACK_RESULTS_DIR,
                                        f'{ATTACK_NAME}_comparative_SEC.png')
        plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

        # --------------------------------------------------------------------------
        # 5. Hyperparameter Effect Plot — ASR vs max_iter at fixed epsilon values
        #    All other parameters fixed — ceteris paribus.
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[m][eps_idx] for m in MAX_ITER_LIST]
            ax.plot(MAX_ITER_LIST, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of max_iter on ASR — {ATTACK_NAME}',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(MAX_ITER_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_path = os.path.join(ATTACK_RESULTS_DIR,
                                f'{ATTACK_NAME}_maxiter_effect.png')
        plt.savefig(effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Hyperparameter effect plot saved at: {effect_path}")

        # --------------------------------------------------------------------------
        # 6. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 7. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:               {mode}")
        print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Max Iter Values:           {MAX_ITER_LIST}")
        print("-" * 50)
        for max_iter, asrs in all_asrs.items():
            print(f"  max_iter={max_iter:2d} — Highest ASR: {asrs[-1]:.2f}% (at max eps)")
        print("=" * 50 + "\n")

## Projected Gradient Descent (PGD) — Error-Specific (Targeted) Attack

This step evaluates PGD in its targeted (error-specific) variant on NN1 across two
targeting strategies (Fixed and LeastLikely), a range of `max_iter` values, and a
range of `num_random_init` values. The step size is set to `eps / max_iter` for each
configuration (ceteris paribus). Individual and comparative Targeted SECs and
hyperparameter effect plots for both `max_iter` and `num_random_init` are saved
alongside a full CSV summary of all results.




In [ ]:
if RUN_PGD_SPECIFIC:
    
    # ==============================================================================
    #  Projected Gradient Descent (PGD) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing PGD Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------
    ATTACK_NAME_BASE   = "PGD_Targeted"
    BATCH_SIZE         = 32
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS = {
        1:  '#1f77b4',
        5:  '#ffbf00',
        10: '#2ca02c',
        20: '#d62728',
        40: '#9467bd'
    }

    RAND_COLORS = {
        0: '#1f77b4',
        1: '#ff7f0e',
        3: '#2ca02c',
        5: '#d62728'
    }

    FIXED_EPSILONS_FOR_EFFECT = [0.01, 0.05, 0.1]
    EPSILON_EFFECT_COLORS     = {
        0.01: '#9467bd',
        0.05: '#8c564b',
        0.1:  '#e377c2'
    }

    # ------------------------------------------------------------------------------
    # 2. Evaluation Loop
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} across "
            f"{len(NUM_RANDOM_INITS)} nri x "
            f"{len(MAX_ITER_LIST)} iters x "
            f"{len(EPSILONS)} epsilons")
        print(f"{'*'*70}")

        results        = []
        all_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}

        for num_random_init in NUM_RANDOM_INITS:
            print(f"\n{'#'*50}")
            print(f"NUM_RANDOM_INIT = {num_random_init} | MODE = {mode}")
            print(f"{'#'*50}")

            for max_iter in MAX_ITER_LIST:
                print(f"\n{'='*50}")
                print(f"MAX_ITER = {max_iter} | NUM_RANDOM_INIT = {num_random_init}")
                print(f"{'='*50}")

                attack_success_rates = []

                for eps in EPSILONS:
                    print(f"\n--- eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init} ---")

                    eps_step = eps / max_iter

                    pgd = ProjectedGradientDescentPyTorch(
                        estimator=art_classifier,
                        norm=np.inf,
                        eps=eps,
                        eps_step=eps_step,
                        max_iter=max_iter,
                        num_random_init=num_random_init,
                        targeted=True,
                        batch_size=BATCH_SIZE,
                        verbose=False
                    )

                    successful_attacks = 0
                    total_processed    = 0
                    start_time         = time.time()

                    for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Processing Batches"):
                        batch_paths    = image_paths[i:i + BATCH_SIZE]
                        batch_true_ids = true_folder_ids[i:i + BATCH_SIZE]

                        batch_tensors   = []
                        batch_filenames = []
                        for path in batch_paths:
                            img = Image.open(path).convert('RGB')
                            batch_tensors.append(preprocess_attack(img).numpy())
                            batch_filenames.append(os.path.basename(path))

                        x_batch = np.stack(batch_tensors)

                        if mode == "LeastLikely":
                            batch_target_indices = get_least_likely_indices(x_batch)
                        elif mode == "Fixed":
                            batch_target_indices = [FIXED_TARGET_CLASS] * len(batch_paths)

                        y_target_one_hot = np.zeros((len(batch_paths), NUM_CLASSES))
                        for k, t_idx in enumerate(batch_target_indices):
                            y_target_one_hot[k, int(t_idx)] = 1.0

                        x_adv        = pgd.generate(x=x_batch, y=y_target_one_hot)
                        preds        = art_classifier.predict(x_adv)
                        pred_indices = np.argmax(preds, axis=1)

                        for j, pred_idx in enumerate(pred_indices):
                            if pred_idx == batch_target_indices[j]:
                                successful_attacks += 1

                        save_adversarial_images(
                            x_adv, batch_true_ids, batch_filenames,
                            f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}", eps
                        )
                        total_processed += len(batch_paths)

                    elapsed = time.time() - start_time
                    asr     = (successful_attacks / total_processed) * 100
                    attack_success_rates.append(asr)

                    print(f"ASR at eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init}: {asr:.2f}%")

                    results.append({
                        'attack_name':         ATTACK_NAME,
                        'target_mode':         mode,
                        'num_random_init':     num_random_init,
                        'max_iter':            max_iter,
                        'eps_step':            round(eps / max_iter, 6),
                        'epsilon':             eps,
                        'successful_attacks':  successful_attacks,
                        'total_images':        total_processed,
                        'attack_success_rate': round(asr, 2),
                        'time_elapsed_s':      round(elapsed, 2)
                    })

                all_asrs[num_random_init][max_iter] = attack_success_rates

                # ------------------------------------------------------------------
                # 3. Individual Targeted SEC
                # ------------------------------------------------------------------
                color = ITER_COLORS[max_iter]

                fig, ax = plt.subplots(figsize=(9, 6))
                ax.plot([0.0, EPSILONS[0]], [0.0, attack_success_rates[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, attack_success_rates,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6,
                        label=f'ASR ({ATTACK_NAME}, max_iter={max_iter}, nri={num_random_init})')
                ax.plot(0.0, 0.0,
                        marker='o', markerfacecolor=color,
                        markeredgecolor='#2ca02c', markeredgewidth=1,
                        markersize=6, linestyle='None', label='Baseline ASR (~0%)')

                ax.set_title(f'Targeted SEC — {ATTACK_NAME}\n'
                            f'(max_iter={max_iter}, num_random_init={num_random_init})',
                            fontsize=16, fontweight='bold', pad=15)
                ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
                ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
                ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
                ax.set_xticks(np.arange(0.0, 0.11, 0.01))
                ax.set_yticks(np.arange(0, 101, 10))
                ax.tick_params(axis='both', labelsize=12)
                ax.set_ylim(-5, 105)
                ax.set_xlim(-0.005, 0.105)
                ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                        label=r'Max Allowable $\epsilon$ (0.1)')
                ax.legend(loc='upper left', fontsize=12)
                plt.tight_layout()

                plot_path = os.path.join(
                    ATTACK_RESULTS_DIR,
                    f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_SEC.png'
                )
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"\n[INFO] Individual SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # 4. Comparative SEC per num_random_init — max_iter varies
        # --------------------------------------------------------------------------
        for num_random_init in NUM_RANDOM_INITS:
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            for max_iter in MAX_ITER_LIST:
                color = ITER_COLORS[max_iter]
                asrs  = all_asrs[num_random_init][max_iter]
                ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, asrs,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6, label=f'max_iter={max_iter}')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} | nri={num_random_init} (max_iter varies)',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_comparative_maxiter_SEC.png'
            )
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Comparative SEC (nri={num_random_init}) saved at: {path}")

        # --------------------------------------------------------------------------
        # 5. Comparative SEC per max_iter — num_random_init varies
        # --------------------------------------------------------------------------
        for max_iter in MAX_ITER_LIST:
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(0.0, 0.0, marker='o', markerfacecolor='black',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            for num_random_init in NUM_RANDOM_INITS:
                color = RAND_COLORS[num_random_init]
                asrs  = all_asrs[num_random_init][max_iter]
                ax.plot([0.0, EPSILONS[0]], [0.0, asrs[0]],
                        linestyle='-', color=color, linewidth=1.5)
                ax.plot(EPSILONS, asrs,
                        marker='o', linestyle='-', color=color,
                        linewidth=1.5, markersize=6, label=f'nri={num_random_init}')

            ax.set_title(f'Targeted SEC — {ATTACK_NAME} | max_iter={max_iter} (nri varies)',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2,
                    label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            path = os.path.join(
                ATTACK_RESULTS_DIR,
                f'{ATTACK_NAME}_iter{max_iter}_comparative_nri_SEC.png'
            )
            plt.savefig(path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] Comparative SEC (max_iter={max_iter}) saved at: {path}")

        # --------------------------------------------------------------------------
        # 6. Hyperparameter Effect Plot — ASR vs max_iter at fixed epsilon
        #    num_random_init fixed at maximum value — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_NRI_FOR_EFFECT  = max(NUM_RANDOM_INITS)

        fig, ax = plt.subplots(figsize=(9, 6))
        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[FIXED_NRI_FOR_EFFECT][m][eps_idx] for m in MAX_ITER_LIST]
            ax.plot(MAX_ITER_LIST, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of max_iter on ASR — {ATTACK_NAME}\n'
                    f'(num_random_init={FIXED_NRI_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Iterations (max_iter)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(MAX_ITER_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_iter_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_maxiter_effect_nri{FIXED_NRI_FOR_EFFECT}.png'
        )
        plt.savefig(effect_iter_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] max_iter effect plot saved at: {effect_iter_path}")

        # --------------------------------------------------------------------------
        # 7. Hyperparameter Effect Plot — ASR vs num_random_init at fixed epsilon
        #    max_iter fixed at maximum value — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_ITER_FOR_EFFECT = max(MAX_ITER_LIST)

        fig, ax = plt.subplots(figsize=(9, 6))
        for fixed_eps in FIXED_EPSILONS_FOR_EFFECT:
            eps_idx     = EPSILONS.index(fixed_eps)
            asrs_at_eps = [all_asrs[nri][FIXED_ITER_FOR_EFFECT][eps_idx] for nri in NUM_RANDOM_INITS]
            ax.plot(NUM_RANDOM_INITS, asrs_at_eps,
                    marker='o', linestyle='-',
                    color=EPSILON_EFFECT_COLORS[fixed_eps],
                    linewidth=1.5, markersize=6,
                    label=rf'$\epsilon$ = {fixed_eps}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--',
                linewidth=1.5, label='Baseline ASR (~0%)')
        ax.set_title(f'Effect of num_random_init on ASR — {ATTACK_NAME}\n'
                    f'(max_iter={FIXED_ITER_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Number of Random Restarts (num_random_init)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(NUM_RANDOM_INITS)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        effect_nri_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_nri_effect_iter{FIXED_ITER_FOR_EFFECT}.png'
        )
        plt.savefig(effect_nri_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] num_random_init effect plot saved at: {effect_nri_path}")

        # --------------------------------------------------------------------------
        # 8. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 9. Attack Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 50)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 50)
        print(f"Target Mode:               {mode}")
        print(f"Evaluated Epsilon Range:   {EPSILONS[0]} to {EPSILONS[-1]}")
        print(f"Max Iter Values:           {MAX_ITER_LIST}")
        print(f"Num Random Init Values:    {NUM_RANDOM_INITS}")
        print("-" * 50)
        for num_random_init in NUM_RANDOM_INITS:
            for max_iter in MAX_ITER_LIST:
                asr_best = all_asrs[num_random_init][max_iter][-1]
                print(f"  nri={num_random_init} | max_iter={max_iter:2d} — "
                    f"Highest ASR: {asr_best:.2f}% (at max eps)")
        print("=" * 50 + "\n")

## Carlini & Wagner L-inf (CW) — Error-Specific (Targeted) Attack

This step evaluates the Carlini & Wagner L-infinity attack in its targeted
(error-specific) variant on NN1. Two targeting strategies are evaluated:
- **Fixed**: every image is pushed toward a single fixed target class (impersonation).
- **LeastLikely**: each image is pushed toward the class assigned the lowest
  probability by the clean model, maximizing the semantic distance of the
  misclassification.

The metric reported is the Attack Success Rate (ASR), defined as the fraction
of images for which the model predicts exactly the intended target class.
Images for which CW exceeds the L-infinity budget are counted as attack failures
and are never excluded from the ASR denominator.

Due to the high computational cost of CW, the 1-per-person subset (100 images)
is used. The attack is evaluated across a grid of `max_iter`, `learning_rate`,
and `confidence`. Targeted SECs, hyperparameter effect plots, and a CSV summary
are saved for each targeting mode.

In [ ]:
if RUN_CW_SPECIFIC:
    # ==============================================================================
    # Carlini & Wagner L-inf (CW) — Error-Specific (Targeted) Attack
    # ==============================================================================

    print("Initializing CW L-inf Targeted Attack Pipeline...")

    # ------------------------------------------------------------------------------
    # 1. Configuration
    # ------------------------------------------------------------------------------

    ATTACK_NAME_BASE   = "CW_Targeted"
    EPSILON_LIMIT      = 0.1
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026  
    TARGET_MODES       = ["LeastLikely"]

    MAX_ITER_LIST      = [1, 5, 10, 20]
    LEARNING_RATE_LIST = [0.001, 0.01, 0.05, 0.1]
    CONFIDENCE_LIST    = [0.0, 0.5, 1.0]

    ITER_COLORS = {
        10:  '#d62728',
        50:  '#8c1a1a',
        100: '#e07070'
    }

    # ------------------------------------------------------------------------------
    # 2. Load 100-image dataset (1 per subject)
    # ------------------------------------------------------------------------------
    print("\nLoading 1-per-person dataset...")

    preprocess_cw = transforms.Compose([
        transforms.Resize((160, 160)),
        transforms.ToTensor()
    ])

    x_data      = []
    image_names = []
    true_ids    = []

    image_files = sorted([
        f for f in os.listdir(DATA_DIR_100)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    if len(image_files) != 100:
        print(f"[WARNING] Found {len(image_files)} images instead of 100.")

    for img_name in image_files:
        img_path = os.path.join(DATA_DIR_100, img_name)
        img      = Image.open(img_path).convert('RGB')
        x_data.append(preprocess_cw(img).numpy())
        image_names.append(img_name)
        true_ids.append(os.path.splitext(img_name)[0].strip())

    x_data = np.array(x_data)
    print(f"Loaded {len(x_data)} images. Shape: {x_data.shape}")

    # ------------------------------------------------------------------------------
    # 3. Evaluation Loop (over target modes)
    # ------------------------------------------------------------------------------
    for mode in TARGET_MODES:
        ATTACK_NAME        = f"{ATTACK_NAME_BASE}_{mode}"
        ATTACK_RESULTS_DIR = os.path.join(RESULTS_DIR, ATTACK_NAME)
        ADV_BASE_DIR       = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
        os.makedirs(ATTACK_RESULTS_DIR, exist_ok=True)
        os.makedirs(ADV_BASE_DIR, exist_ok=True)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} — "
            f"{len(MAX_ITER_LIST)} max_iter x "
            f"{len(LEARNING_RATE_LIST)} lr x "
            f"{len(CONFIDENCE_LIST)} confidence combinations")
        print(f"{'*'*70}")

        results        = []
        all_combo_data = []  # Stores (label, max_iter, mean_linf, asr) for plots

        total_combinations = (len(MAX_ITER_LIST) * len(LEARNING_RATE_LIST) * len(CONFIDENCE_LIST))
        current_combo = 1

        # Pre-compute target indices for LeastLikely mode once per mode
        # to avoid redundant forward passes inside the combination loop
        if mode == "LeastLikely":
            print("\nPre-computing least likely target indices...")
            least_likely_targets = get_least_likely_indices(x_data)
            print(f"Least likely targets computed for {len(least_likely_targets)} images.")

        for max_iter in MAX_ITER_LIST:
            for lr in LEARNING_RATE_LIST:
                for conf in CONFIDENCE_LIST:
                    print(f"\n[{current_combo}/{total_combinations}] "
                        f"max_iter={max_iter} | lr={lr} | confidence={conf} | "
                        f"mode={mode}")

                    lr_str         = str(lr).replace('.', '_')
                    conf_str       = str(conf).replace('.', '_')
                    combo_dir_name = f"iter_{max_iter}_lr_{lr_str}_c_{conf_str}"
                    adv_combo_dir  = os.path.join(ADV_BASE_DIR, combo_dir_name)
                    os.makedirs(adv_combo_dir, exist_ok=True)

                    # Build one-hot target array for this combination
                    y_target_one_hot = np.zeros((len(x_data), NUM_CLASSES))
                    batch_target_indices = []

                    for k in range(len(x_data)):
                        if mode == "Fixed":
                            target_idx = FIXED_TARGET_CLASS
                        else:
                            target_idx = least_likely_targets[k]

                        batch_target_indices.append(target_idx)
                        y_target_one_hot[k, int(target_idx)] = 1.0

                    attack_cw = CarliniLInfMethod(
                        classifier=art_classifier,
                        targeted=True,
                        max_iter=max_iter,
                        learning_rate=lr,
                        confidence=conf,
                        verbose=False
                    )

                    start_time = time.time()

                    # Generate adversarial examples one at a time
                    x_adv_list = []
                    for i in range(len(x_data)):
                        single_adv = attack_cw.generate(
                            x=x_data[i:i+1],
                            y=y_target_one_hot[i:i+1]
                        )
                        x_adv_list.append(single_adv)
                        if (i + 1) % 10 == 0:
                            print(f"  Progress: {i+1}/{len(x_data)} images generated...")

                    x_adv_raw = np.concatenate(x_adv_list, axis=0)

                    # ------------------------------------------------------------------
                    # Evaluate ASR on ALL images — budget failures count as failures
                    # ------------------------------------------------------------------
                    perturbation_raw = x_adv_raw - x_data
                    preds_adv        = np.argmax(art_classifier.predict(x_adv_raw), axis=1)

                    successful_attacks = 0
                    saved_count        = 0
                    l_inf_list         = []

                    for i in range(len(x_adv_raw)):
                        l_inf_img = np.max(np.abs(perturbation_raw[i]))

                        if l_inf_img > EPSILON_LIMIT + 1e-5:
                            # Budget exceeded — attack failed, counts as unsuccessful
                            continue

                        # Valid perturbation within budget
                        adv_clipped = np.clip(x_adv_raw[i], 0.0, 1.0)
                        actual_pert = adv_clipped - x_data[i]
                        l_inf_list.append(np.max(np.abs(actual_pert)))

                        # Save adversarial image
                        adv_save = np.transpose(adv_clipped, (1, 2, 0))
                        adv_save = (adv_save * 255.0).astype(np.uint8)
                        Image.fromarray(adv_save).save(
                            os.path.join(adv_combo_dir, image_names[i])
                        )
                        saved_count += 1

                        # Check if attack succeeded — model predicts intended target
                        if preds_adv[i] == batch_target_indices[i]:
                            successful_attacks += 1

                    elapsed        = time.time() - start_time
                    # ASR denominator is always total images, not just saved ones
                    asr            = (successful_attacks / len(x_data)) * 100
                    mean_linf      = float(np.mean(l_inf_list)) if l_inf_list else 0.0
                    failed_budget  = len(x_data) - saved_count

                    print(f"  Saved: {saved_count}/100 | " f"Budget exceeded: {failed_budget}/100")
                    print(f"  ASR: {asr:.2f}% | " f"Mean L-inf applied: {mean_linf:.4f} | " f"Time: {elapsed:.1f}s")

                    results.append({
                        'attack_name':       ATTACK_NAME,
                        'target_mode':       mode,
                        'max_iter':          max_iter,
                        'learning_rate':     lr,
                        'confidence':        conf,
                        'attack_success_rate': round(asr, 2),
                        'successful_attacks':  successful_attacks,
                        'saved_images':      saved_count,
                        'failed_budget':     failed_budget,
                        'total_images':      len(x_data),
                        'mean_linf_applied': round(mean_linf, 4),
                        'time_elapsed_s':    round(elapsed, 2)
                    })

                    all_combo_data.append({
                        'label':    f'iter={max_iter}, lr={lr}, c={conf}',
                        'max_iter': max_iter,
                        'mean_linf': mean_linf,
                        'asr':      asr
                    })

                    current_combo += 1

        # --------------------------------------------------------------------------
        # 4. Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # 5. Targeted SEC — ASR vs Mean L-inf Applied
        #    X-axis: mean L-inf perturbation actually applied.
        #    One curve per max_iter value, each point is one (lr, confidence) combo.
        #    Ceteris paribus: for each curve max_iter is fixed, lr and confidence vary.
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--', linewidth=1.5, label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            color  = ITER_COLORS[max_iter]
            combos = [c for c in all_combo_data if c['max_iter'] == max_iter]
            combos.sort(key=lambda c: c['mean_linf'])
            x_vals = [c['mean_linf'] for c in combos]
            y_vals = [c['asr']       for c in combos]

            ax.plot(x_vals, y_vals,
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.axvline(x=EPSILON_LIMIT, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')

        ax.set_title(f'Targeted Security Evaluation Curve — {ATTACK_NAME}',fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Mean L-inf Perturbation Applied', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        sec_path = os.path.join(ATTACK_RESULTS_DIR, f'{ATTACK_NAME}_SEC.png')
        plt.savefig(sec_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Targeted SEC saved at: {sec_path}")

        # --------------------------------------------------------------------------
        # 6. Hyperparameter Effect Plot — ASR vs learning_rate at fixed max_iter
        #    confidence fixed at 0.0 — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_CONF_FOR_EFFECT = 0.0

        fig, ax = plt.subplots(figsize=(9, 6))

        for max_iter in MAX_ITER_LIST:
            color  = ITER_COLORS[max_iter]
            subset = df_results[
                (df_results['max_iter'] == max_iter) &
                (df_results['confidence'] == FIXED_CONF_FOR_EFFECT)
            ].sort_values('learning_rate')

            ax.plot(subset['learning_rate'].tolist(),
                    subset['attack_success_rate'].tolist(),
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--', linewidth=1.5, label='Baseline ASR (~0%)')

        ax.set_title(f'Effect of learning_rate on ASR — {ATTACK_NAME}\n'
                    f'(confidence={FIXED_CONF_FOR_EFFECT})',
                    fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Learning Rate', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xscale('log')
        ax.set_xticks(LEARNING_RATE_LIST)
        ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        lr_effect_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_lr_effect_conf{FIXED_CONF_FOR_EFFECT}.png'
        )
        plt.savefig(lr_effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Learning rate effect plot saved at: {lr_effect_path}")

        # --------------------------------------------------------------------------
        # 7. Hyperparameter Effect Plot — ASR vs confidence at fixed max_iter
        #    learning_rate fixed at 0.01 — ceteris paribus.
        # --------------------------------------------------------------------------
        FIXED_LR_FOR_EFFECT = 0.01

        fig, ax = plt.subplots(figsize=(9, 6))

        for max_iter in MAX_ITER_LIST:
            color  = ITER_COLORS[max_iter]
            subset = df_results[
                (df_results['max_iter'] == max_iter) &
                (df_results['learning_rate'] == FIXED_LR_FOR_EFFECT)
            ].sort_values('confidence')

            ax.plot(subset['confidence'].tolist(),
                    subset['attack_success_rate'].tolist(),
                    marker='o', linestyle='-', color=color,
                    linewidth=1.5, markersize=6,
                    label=f'max_iter={max_iter}')

        ax.axhline(y=0.0, color='#2ca02c', linestyle='--', linewidth=1.5, label='Baseline ASR (~0%)')

        ax.set_title(f'Effect of confidence on ASR — {ATTACK_NAME}\n' f'(learning_rate={FIXED_LR_FOR_EFFECT})', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel('Confidence', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.set_xticks(CONFIDENCE_LIST)
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        conf_effect_path = os.path.join(
            ATTACK_RESULTS_DIR,
            f'{ATTACK_NAME}_conf_effect_lr{FIXED_LR_FOR_EFFECT}.png'
        )
        plt.savefig(conf_effect_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Confidence effect plot saved at: {conf_effect_path}")

        # --------------------------------------------------------------------------
        # 8. Attack Summary
        # --------------------------------------------------------------------------
        best_row = df_results.loc[df_results['attack_success_rate'].idxmax()]

        print("\n" + "=" * 60)
        print(f"{ATTACK_NAME} ATTACK SUMMARY REPORT")
        print("=" * 60)
        print(f"Target Mode:           {mode}")
        print(f"Dataset:               1-per-person subset (100 images)")
        print(f"Hyperparameter Grid:   {total_combinations} combinations evaluated")
        print("-" * 60)
        print(f"Best Configuration (highest ASR):")
        print(f"  max_iter:      {best_row['max_iter']}")
        print(f"  learning_rate: {best_row['learning_rate']}")
        print(f"  confidence:    {best_row['confidence']}")
        print(f"  ASR:           {best_row['attack_success_rate']:.2f}%")
        print(f"  Mean L-inf:    {best_row['mean_linf_applied']:.4f}")
        print("=" * 60 + "\n")

# Step 5: Transferability Evaluation — ResNet-50 (NN2)

This section evaluates whether adversarial examples generated against NN1
(InceptionResnetV1) transfer to NN2 (ResNet-50), i.e., whether they cause
misclassification on a model they were not optimized for.

For error-generic attacks, transferability is measured as the drop in NN2
accuracy relative to its clean baseline. For error-specific attacks,
transferability is measured as the Attack Success Rate on NN2 — the fraction
of images for which NN2 predicts exactly the target class that was imposed
during generation against NN1.

Adversarial images are preprocessed for NN2 using nearest-neighbor interpolation
to preserve the L-infinity perturbation as faithfully as possible during the
required resize from 160x160 to 224x224.

## Step 5.1: Transferability — FGSM (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by FGSM against NN1,
across all epsilon values. A comparative SEC overlays NN1 and NN2 accuracy curves
to visualize the transferability gap.

In [ ]:
if RUN_FGSM_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.1: Transferability — FGSM (Error-Generic)
    # ==============================================================================

    print("Evaluating FGSM transferability on NN2...")

    ATTACK_NAME        = "FGSM"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    BATCH_SIZE         = 32

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    ADV_ATTACK_DIR = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)

    # Load NN1 results CSV to retrieve NN1 accuracies for the comparative SEC
    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results      = []
    nn2_accuracies = []
    nn1_accuracies = []

    for eps in EPSILONS:
        eps_folder    = f"eps_{eps:.3f}".replace('.', '_')
        adv_eps_dir   = os.path.join(ADV_ATTACK_DIR, eps_folder)

        if not os.path.exists(adv_eps_dir):
            print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
            continue

        print(f"\n--- Evaluating Epsilon: {eps:.3f} ---")

        correct_nn2    = 0
        total_processed = 0
        start_time      = time.time()

        for class_id in os.listdir(adv_eps_dir):
            class_folder = os.path.join(adv_eps_dir, class_id)
            if not os.path.isdir(class_folder):
                continue

            true_name  = id_to_name.get(class_id.strip(), "Unknown")
            true_clean = normalize_label(true_name)

            for img_name in os.listdir(class_folder):
                if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue

                img_path = os.path.join(class_folder, img_name)
                try:
                    img        = Image.open(img_path).convert('RGB')
                    tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                    with torch.no_grad():
                        pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                    pred_clean_2 = clean_pred_label(pred_idx_2)

                    if pred_clean_2 == true_clean:
                        correct_nn2 += 1

                    total_processed += 1

                except Exception as e:
                    print(f"Error processing {img_path}: {e}")

        elapsed    = time.time() - start_time
        acc_nn2    = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0
        nn2_accuracies.append(acc_nn2)

        # Retrieve corresponding NN1 accuracy from saved CSV
        nn1_row = df_nn1[df_nn1['epsilon'] == eps]
        acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0
        nn1_accuracies.append(acc_nn1)

        print(f"NN2 Accuracy at eps={eps:.3f}: {acc_nn2:.2f}%  |  NN1: {acc_nn1:.2f}%")

        results.append({
            'attack_name':   ATTACK_NAME,
            'epsilon':       eps,
            'nn1_accuracy':  round(acc_nn1, 2),
            'nn2_accuracy':  round(acc_nn2, 2),
            'nn2_correct':   correct_nn2,
            'total_images':  total_processed,
            'time_elapsed_s': round(elapsed, 2)
        })

    # ------------------------------------------------------------------------------
    # Comparative SEC — NN1 vs NN2
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC_1000,
            marker='o', markerfacecolor='#1f77b4',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
    ax.plot(0.0, BASELINE_ACC_NN2_1000,
            marker='o', markerfacecolor='#ff7f0e',
            markeredgecolor='#2ca02c', markeredgewidth=1,
            markersize=6, linestyle='None',
            label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

    ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accuracies[0]],
            linestyle='-', color='#1f77b4', linewidth=1.5)
    ax.plot(EPSILONS, nn1_accuracies,
            marker='o', linestyle='-', color='#1f77b4',
            linewidth=1.5, markersize=6, label=f'NN1 Under Attack')

    ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accuracies[0]],
            linestyle='-', color='#ff7f0e', linewidth=1.5)
    ax.plot(EPSILONS, nn2_accuracies,
            marker='o', linestyle='-', color='#ff7f0e',
            linewidth=1.5, markersize=6, label=f'NN2 Transferability')

    ax.set_title(f'Transferability SEC — {ATTACK_NAME} (NN1 vs NN2)', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=12)
    plt.tight_layout()

    plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_SEC.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 55)
    print(f"FGSM TRANSFERABILITY SUMMARY")
    print("=" * 55)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    print(f"{'Epsilon':<10} {'NN1 Acc':>10} {'NN2 Acc':>10} {'Transfer Gap':>14}")
    print("-" * 55)
    for eps, a1, a2 in zip(EPSILONS, nn1_accuracies, nn2_accuracies):
        print(f"{eps:<10.3f} {a1:>10.2f}% {a2:>10.2f}% {(a1 - a2):>+13.2f}%")
    print("=" * 55 + "\n")

## Step 5.2: Transferability — BIM (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by BIM against NN1,
across all combinations of epsilon and max_iter. A comparative SEC is generated
for each max_iter value, overlaying NN1 and NN2 accuracy curves.
A summary table reports the transferability gap for each configuration.

In [ ]:
if RUN_BIM_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.2: Transferability — BIM (Error-Generic)
    # ==============================================================================

    print("Evaluating BIM transferability on NN2...")

    ATTACK_NAME    = "BIM"
    EPSILONS       = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST  = [1, 5, 10, 20]
    BATCH_SIZE     = 32

    ITER_COLORS_NN1 = {1: '#1f77b4', 5: '#ff7f0e', 10: '#2ca02c', 20: '#d62728'}
    ITER_COLORS_NN2 = {1: '#aec7e8', 5: '#ffbb78', 10: '#98df8a', 20: '#ff9896'}

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results            = []
    all_nn1_accuracies = {}
    all_nn2_accuracies = {}

    for max_iter in MAX_ITER_LIST:
        folder_name  = f"{ATTACK_NAME}_iter{max_iter}"
        adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

        if not os.path.exists(adv_iter_dir):
            print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
            continue

        print(f"\n{'='*50}")
        print(f"MAX_ITER = {max_iter}")
        print(f"{'='*50}")

        nn1_accs = []
        nn2_accs = []

        for eps in EPSILONS:
            eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
            adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

            if not os.path.exists(adv_eps_dir):
                print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                continue

            print(f"\n--- Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

            correct_nn2     = 0
            total_processed = 0
            start_time      = time.time()

            for class_id in os.listdir(adv_eps_dir):
                class_folder = os.path.join(adv_eps_dir, class_id)
                if not os.path.isdir(class_folder):
                    continue

                true_name  = id_to_name.get(class_id.strip(), "Unknown")
                true_clean = normalize_label(true_name)

                for img_name in os.listdir(class_folder):
                    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue

                    img_path = os.path.join(class_folder, img_name)
                    try:
                        img        = Image.open(img_path).convert('RGB')
                        tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                        with torch.no_grad():
                            pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                        if clean_pred_label(pred_idx_2) == true_clean:
                            correct_nn2 += 1

                        total_processed += 1

                    except Exception as e:
                        print(f"Error processing {img_path}: {e}")

            elapsed = time.time() - start_time
            acc_nn2 = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

            nn1_row = df_nn1[(df_nn1['epsilon'] == eps) & (df_nn1['max_iter'] == max_iter)]
            acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0

            nn1_accs.append(acc_nn1)
            nn2_accs.append(acc_nn2)

            print(f"NN2 Accuracy: {acc_nn2:.2f}%  |  NN1: {acc_nn1:.2f}%")

            results.append({
                'attack_name':    ATTACK_NAME,
                'max_iter':       max_iter,
                'epsilon':        eps,
                'nn1_accuracy':   round(acc_nn1, 2),
                'nn2_accuracy':   round(acc_nn2, 2),
                'nn2_correct':    correct_nn2,
                'total_images':   total_processed,
                'time_elapsed_s': round(elapsed, 2)
            })

        all_nn1_accuracies[max_iter] = nn1_accs
        all_nn2_accuracies[max_iter] = nn2_accs

        # --------------------------------------------------------------------------
        # Individual Comparative SEC for this max_iter
        # --------------------------------------------------------------------------
        c_nn1 = ITER_COLORS_NN1[max_iter]
        c_nn2 = ITER_COLORS_NN2[max_iter]

        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, BASELINE_ACC_1000,
                marker='o', markerfacecolor=c_nn1,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
        ax.plot(0.0, BASELINE_ACC_NN2_1000,
                marker='o', markerfacecolor=c_nn2,
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None',
                label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accs[0]],
                linestyle='-', color=c_nn1, linewidth=1.5)
        ax.plot(EPSILONS, nn1_accs,
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=6,
                label=f'NN1 Under Attack (max_iter={max_iter})')

        ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accs[0]],
                linestyle='--', color=c_nn2, linewidth=1.5)
        ax.plot(EPSILONS, nn2_accs,
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=6,
                label=f'NN2 Transferability (max_iter={max_iter})')

        ax.set_title(f'Transferability SEC — {ATTACK_NAME} (max_iter={max_iter})', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Model Accuracy (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper right', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_iter{max_iter}_transferability_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Comparative overview — all max_iter, NN1 vs NN2
    # ------------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 6))

    ax.plot(0.0, BASELINE_ACC_1000,
            marker='o', color='black', markersize=6,
            linestyle='None', label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
    ax.plot(0.0, BASELINE_ACC_NN2_1000,
            marker='s', color='gray', markersize=6,
            linestyle='None', label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

    for max_iter in MAX_ITER_LIST:
        if max_iter not in all_nn1_accuracies:
            continue
        c_nn1 = ITER_COLORS_NN1[max_iter]
        c_nn2 = ITER_COLORS_NN2[max_iter]

        ax.plot([0.0, EPSILONS[0]],
                [BASELINE_ACC_1000, all_nn1_accuracies[max_iter][0]],
                linestyle='-', color=c_nn1, linewidth=1.5)
        ax.plot(EPSILONS, all_nn1_accuracies[max_iter],
                marker='o', linestyle='-', color=c_nn1,
                linewidth=1.5, markersize=5,
                label=f'NN1 iter={max_iter}')

        ax.plot([0.0, EPSILONS[0]],
                [BASELINE_ACC_NN2_1000, all_nn2_accuracies[max_iter][0]],
                linestyle='--', color=c_nn2, linewidth=1.5)
        ax.plot(EPSILONS, all_nn2_accuracies[max_iter],
                marker='s', linestyle='--', color=c_nn2,
                linewidth=1.5, markersize=5,
                label=f'NN2 iter={max_iter}')

    ax.set_title(f'Transferability SEC — {ATTACK_NAME} (All max_iter, Comparative)', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
    ax.set_ylabel('Model Accuracy (%)', fontsize=14)
    ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
    ax.set_xticks(np.arange(0.0, 0.11, 0.01))
    ax.set_yticks(np.arange(0, 101, 10))
    ax.tick_params(axis='both', labelsize=12)
    ax.set_ylim(-5, 105)
    ax.set_xlim(-0.005, 0.105)
    ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()

    comparative_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_comparative_SEC.png')
    plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n[INFO] Comparative transferability SEC saved at: {comparative_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print(f"BIM TRANSFERABILITY SUMMARY")
    print("=" * 60)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    for max_iter in MAX_ITER_LIST:
        if max_iter not in all_nn1_accuracies:
            continue
        a1 = all_nn1_accuracies[max_iter][-1]
        a2 = all_nn2_accuracies[max_iter][-1]
        print(f"  max_iter={max_iter:2d} @ eps=0.1 — NN1: {a1:.2f}%  NN2: {a2:.2f}%  " f"Gap: {(a1-a2):+.2f}%")
    print("=" * 60 + "\n")

## Step 5.3: Transferability — PGD (Error-Generic)

Evaluates NN2 accuracy on adversarial examples generated by PGD against NN1,
across all combinations of epsilon, max_iter, and num_random_init.
Individual comparative SECs are generated for each (max_iter, num_random_init)
combination. A summary table reports the transferability gap for the strongest
configuration (max max_iter, max num_random_init).

In [ ]:
if RUN_PGD_TRANSFERABILITY_GENERIC:
    # ==============================================================================
    # STEP 5.3: Transferability — PGD (Error-Generic)
    # ==============================================================================

    print("Evaluating PGD transferability on NN2...")

    ATTACK_NAME      = "PGD"
    EPSILONS         = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST    = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS = [0, 1, 3, 5]
    BATCH_SIZE       = 32

    ITER_COLORS_NN1 = {
        1: '#1f77b4', 5: '#ffbf00', 10: '#2ca02c', 20: '#d62728', 40: '#9467bd'
    }
    ITER_COLORS_NN2 = {
        1: '#aec7e8', 5: '#ffe599', 10: '#98df8a', 20: '#ff9896', 40: '#c5b0d5'
    }

    TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
    os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

    nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")
    df_nn1           = pd.read_csv(nn1_results_path)

    results            = []
    all_nn1_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}
    all_nn2_accuracies = {nri: {} for nri in NUM_RANDOM_INITS}

    for num_random_init in NUM_RANDOM_INITS:
        for max_iter in MAX_ITER_LIST:
            folder_name  = f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}"
            adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

            if not os.path.exists(adv_iter_dir):
                print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                continue

            print(f"\n{'='*50}")
            print(f"NUM_RANDOM_INIT = {num_random_init} | MAX_ITER = {max_iter}")
            print(f"{'='*50}")

            nn1_accs = []
            nn2_accs = []

            for eps in EPSILONS:
                eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                if not os.path.exists(adv_eps_dir):
                    print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                    continue

                correct_nn2     = 0
                total_processed = 0
                start_time      = time.time()

                for class_id in os.listdir(adv_eps_dir):
                    class_folder = os.path.join(adv_eps_dir, class_id)
                    if not os.path.isdir(class_folder):
                        continue

                    true_name  = id_to_name.get(class_id.strip(), "Unknown")
                    true_clean = normalize_label(true_name)

                    for img_name in os.listdir(class_folder):
                        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                            continue

                        img_path = os.path.join(class_folder, img_name)
                        try:
                            img        = Image.open(img_path).convert('RGB')
                            tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                            with torch.no_grad():
                                pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                            if clean_pred_label(pred_idx_2) == true_clean:
                                correct_nn2 += 1

                            total_processed += 1

                        except Exception as e:
                            print(f"Error processing {img_path}: {e}")

                elapsed = time.time() - start_time
                acc_nn2 = (correct_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                nn1_row = df_nn1[
                    (df_nn1['epsilon'] == eps) &
                    (df_nn1['max_iter'] == max_iter) &
                    (df_nn1['num_random_init'] == num_random_init)
                ]
                acc_nn1 = nn1_row['accuracy'].values[0] if len(nn1_row) > 0 else 0.0

                nn1_accs.append(acc_nn1)
                nn2_accs.append(acc_nn2)

                print(f"eps={eps:.3f} | NN1: {acc_nn1:.2f}%  NN2: {acc_nn2:.2f}%")

                results.append({
                    'attack_name':      ATTACK_NAME,
                    'num_random_init':  num_random_init,
                    'max_iter':         max_iter,
                    'epsilon':          eps,
                    'nn1_accuracy':     round(acc_nn1, 2),
                    'nn2_accuracy':     round(acc_nn2, 2),
                    'nn2_correct':      correct_nn2,
                    'total_images':     total_processed,
                    'time_elapsed_s':   round(elapsed, 2)
                })

            all_nn1_accuracies[num_random_init][max_iter] = nn1_accs
            all_nn2_accuracies[num_random_init][max_iter] = nn2_accs

            # ----------------------------------------------------------------------
            # Individual Comparative SEC for this (nri, max_iter) combination
            # ----------------------------------------------------------------------
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot(0.0, BASELINE_ACC_1000,
                    marker='o', markerfacecolor=c_nn1,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'NN1 Baseline ({BASELINE_ACC_1000:.2f}%)')
            ax.plot(0.0, BASELINE_ACC_NN2_1000,
                    marker='o', markerfacecolor=c_nn2,
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None',
                    label=f'NN2 Baseline ({BASELINE_ACC_NN2_1000:.2f}%)')

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_1000, nn1_accs[0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, nn1_accs,
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 (nri={num_random_init}, iter={max_iter})')

            ax.plot([0.0, EPSILONS[0]], [BASELINE_ACC_NN2_1000, nn2_accs[0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, nn2_accs,
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6,
                    label=f'NN2 (nri={num_random_init}, iter={max_iter})')

            ax.set_title(f'Transferability SEC — {ATTACK_NAME}\n'
                        f'(nri={num_random_init}, max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Model Accuracy (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper right', fontsize=11)
            plt.tight_layout()

            plot_path = os.path.join(
                TRANSFER_RESULTS_DIR,
                f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_transferability_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] SEC saved at: {plot_path}")

    # ------------------------------------------------------------------------------
    # Save Results CSV
    # ------------------------------------------------------------------------------
    df_results = pd.DataFrame(results)
    csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
    df_results.to_csv(csv_path, index=False)
    print(f"\n[INFO] Results CSV saved at: {csv_path}")

    # ------------------------------------------------------------------------------
    # Summary — strongest configuration only
    # ------------------------------------------------------------------------------
    best_nri  = max(NUM_RANDOM_INITS)
    best_iter = max(MAX_ITER_LIST)

    print("\n" + "=" * 60)
    print(f"PGD TRANSFERABILITY SUMMARY (Strongest Config: nri={best_nri}, iter={best_iter})")
    print("=" * 60)
    print(f"NN1 Baseline: {BASELINE_ACC_1000:.2f}%  |  NN2 Baseline: {BASELINE_ACC_NN2_1000:.2f}%")
    if best_nri in all_nn1_accuracies and best_iter in all_nn1_accuracies[best_nri]:
        a1 = all_nn1_accuracies[best_nri][best_iter][-1]
        a2 = all_nn2_accuracies[best_nri][best_iter][-1]
        print(f"  @ eps=0.1 — NN1: {a1:.2f}%  NN2: {a2:.2f}%  Gap: {(a1-a2):+.2f}%")
    print("=" * 60 + "\n")

## Step 5.4: Transferability — FGSM Targeted (Error-Specific)

Evaluates whether adversarial examples generated by FGSM Targeted against NN1
cause NN2 to predict the same intended target class. The Attack Success Rate (ASR)
on NN2 is compared against the ASR on NN1 via a comparative targeted SEC.
Both targeting modes (Fixed and LeastLikely) are evaluated independently.

In [ ]:
if RUN_FGSM_TRANSFERABILITY_SPECIFIC:
    # ==============================================================================
    # STEP 5.4: Transferability — FGSM Targeted (Error-Specific)
    # ==============================================================================

    print("Evaluating FGSM Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "FGSM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026
    BATCH_SIZE         = 32
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

        ADV_ATTACK_DIR   = os.path.join(ADVERSARIAL_DATA_DIR, ATTACK_NAME)
        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME, f"{ATTACK_NAME}_results.csv")

        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        results       = []
        nn1_asrs      = []
        nn2_asrs      = []

        for eps in EPSILONS:
            eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
            adv_eps_dir = os.path.join(ADV_ATTACK_DIR, eps_folder)

            if not os.path.exists(adv_eps_dir):
                print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                continue

            print(f"\n--- Epsilon: {eps:.3f} | Mode: {mode} ---")

            successful_nn2  = 0
            total_processed = 0
            start_time      = time.time()

            for class_id in os.listdir(adv_eps_dir):
                class_folder = os.path.join(adv_eps_dir, class_id)
                if not os.path.isdir(class_folder):
                    continue

                for img_name in os.listdir(class_folder):
                    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue

                    img_path = os.path.join(class_folder, img_name)
                    try:
                        img        = Image.open(img_path).convert('RGB')
                        tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                        with torch.no_grad():
                            pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                        # For Fixed mode the target is always FIXED_TARGET_CLASS.
                        # For LeastLikely the target was computed per-image during generation
                        # and is not stored — Fixed target is used as proxy for transferability
                        # measurement, which remains consistent with the NN1 evaluation metric.
                        if mode == "Fixed":
                            target = FIXED_TARGET_CLASS
                        else:
                            # LeastLikely: re-compute target from the clean image via NN2
                            img_clean  = Image.open(
                                os.path.join(DATA_DIR_1000, class_id, img_name.replace('.png', '.jpg'))).convert('RGB')
                            t_clean    = preprocess_nn2(img_clean).unsqueeze(0).to(device)
                            with torch.no_grad():
                                preds_clean = nn2(t_clean)[0].detach().cpu().numpy()
                            target = int(np.argmin(preds_clean))

                        if pred_idx_2 == target:
                            successful_nn2 += 1

                        total_processed += 1

                    except Exception as e:
                        print(f"Error processing {img_path}: {e}")

            elapsed    = time.time() - start_time
            asr_nn2    = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0
            nn2_asrs.append(asr_nn2)

            nn1_row = df_nn1[df_nn1['epsilon'] == eps]
            asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0
            nn1_asrs.append(asr_nn1)

            print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

            results.append({
                'attack_name':   ATTACK_NAME,
                'target_mode':   mode,
                'epsilon':       eps,
                'nn1_asr':       round(asr_nn1, 2),
                'nn2_asr':       round(asr_nn2, 2),
                'nn2_successful': successful_nn2,
                'total_images':  total_processed,
                'time_elapsed_s': round(elapsed, 2)
            })

        # --------------------------------------------------------------------------
        # Comparative Targeted SEC — NN1 ASR vs NN2 ASR
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                linestyle='-', color='#d62728', linewidth=1.5)
        ax.plot(EPSILONS, nn1_asrs,
                marker='o', linestyle='-', color='#d62728',
                linewidth=1.5, markersize=6, label='NN1 ASR (Attack Source)')

        ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                linestyle='--', color='#ff9896', linewidth=1.5)
        ax.plot(EPSILONS, nn2_asrs,
                marker='s', linestyle='--', color='#ff9896',
                linewidth=1.5, markersize=6, label='NN2 ASR (Transferability)')

        ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=12)
        plt.tight_layout()

        plot_path = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_SEC.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Transferability SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 55)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY")
        print("=" * 55)
        print(f"{'Epsilon':<10} {'NN1 ASR':>10} {'NN2 ASR':>10} {'Gap':>10}")
        print("-" * 55)
        for eps, a1, a2 in zip(EPSILONS, nn1_asrs, nn2_asrs):
            print(f"{eps:<10.3f} {a1:>10.2f}% {a2:>10.2f}% {(a2-a1):>+9.2f}%")
        print("=" * 55 + "\n")

## Step 5.5: Transferability — BIM Targeted (Error-Specific)

Evaluates whether adversarial examples generated by BIM Targeted against NN1
cause NN2 to predict the same intended target class. The Attack Success Rate (ASR)
on NN2 is compared against the ASR on NN1 via comparative targeted SECs, one per
max_iter value. Both targeting modes (Fixed and LeastLikely) are evaluated
independently. A CSV summary of all results is saved for each mode.

In [19]:
if RUN_BIM_TRANSFERABILITY_SPECIFIC:  
    
    # ==============================================================================
    # STEP 5.5: Transferability — BIM Targeted (Error-Specific)
    # ==============================================================================
    print("Evaluating BIM Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "BIM_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026
    BATCH_SIZE         = 32
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS_NN1 = {1: '#d62728', 5: '#8c1a1a', 10: '#e07070', 20: '#a83232'}
    ITER_COLORS_NN2 = {1: '#ff9896', 5: '#ffbcba', 10: '#ffd7d6', 20: '#ffe8e8'}

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME,
                                        f"{ATTACK_NAME}_results.csv")
        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} transferability across " f"{len(MAX_ITER_LIST)} max_iter values x {len(EPSILONS)} epsilon values")
        print(f"{'*'*70}")

        results            = []
        all_nn1_asrs       = {}
        all_nn2_asrs       = {}

        for max_iter in MAX_ITER_LIST:
            folder_name  = f"{ATTACK_NAME}_iter{max_iter}"
            adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

            if not os.path.exists(adv_iter_dir):
                print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                continue

            print(f"\n{'='*50}")
            print(f"MAX_ITER = {max_iter} | MODE = {mode}")
            print(f"{'='*50}")

            nn1_asrs = []
            nn2_asrs = []

            for eps in EPSILONS:
                eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                if not os.path.exists(adv_eps_dir):
                    print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                    continue

                print(f"\n--- Epsilon: {eps:.3f} | max_iter: {max_iter} ---")

                successful_nn2  = 0
                total_processed = 0
                start_time      = time.time()

                for class_id in os.listdir(adv_eps_dir):
                    class_folder = os.path.join(adv_eps_dir, class_id)
                    if not os.path.isdir(class_folder):
                        continue

                    for img_name in os.listdir(class_folder):
                        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                            continue

                        img_path = os.path.join(class_folder, img_name)
                        try:
                            img        = Image.open(img_path).convert('RGB')
                            tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                            with torch.no_grad():
                                pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                            if mode == "Fixed":
                                target = FIXED_TARGET_CLASS
                            else:
                                # LeastLikely: re-compute target from clean image via NN2
                                clean_path = os.path.join(
                                    DATA_DIR_1000, class_id,
                                    img_name.replace('.png', '.jpg')
                                )
                                img_clean  = Image.open(clean_path).convert('RGB')
                                t_clean    = preprocess_nn2(img_clean).unsqueeze(0).to(device)
                                with torch.no_grad():
                                    preds_clean = nn2(t_clean)[0].detach().cpu().numpy()
                                target = int(np.argmin(preds_clean))

                            if pred_idx_2 == target:
                                successful_nn2 += 1

                            total_processed += 1

                        except Exception as e:
                            print(f"Error processing {img_path}: {e}")

                elapsed = time.time() - start_time
                asr_nn2 = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                nn1_row = df_nn1[
                    (df_nn1['epsilon'] == eps) &
                    (df_nn1['max_iter'] == max_iter)
                ]
                asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0

                nn1_asrs.append(asr_nn1)
                nn2_asrs.append(asr_nn2)

                print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

                results.append({
                    'attack_name':    ATTACK_NAME,
                    'target_mode':    mode,
                    'max_iter':       max_iter,
                    'epsilon':        eps,
                    'nn1_asr':        round(asr_nn1, 2),
                    'nn2_asr':        round(asr_nn2, 2),
                    'nn2_successful': successful_nn2,
                    'total_images':   total_processed,
                    'time_elapsed_s': round(elapsed, 2)
                })

            all_nn1_asrs[max_iter] = nn1_asrs
            all_nn2_asrs[max_iter] = nn2_asrs

            # ----------------------------------------------------------------------
            # Individual Comparative Targeted SEC for this max_iter
            # ----------------------------------------------------------------------
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            fig, ax = plt.subplots(figsize=(9, 6))

            ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                    markeredgecolor='#2ca02c', markeredgewidth=1,
                    markersize=6, linestyle='None', label='Baseline ASR (~0%)')

            ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, nn1_asrs,
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=6,
                    label=f'NN1 ASR (max_iter={max_iter})')

            ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, nn2_asrs,
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=6,
                    label=f'NN2 ASR (max_iter={max_iter})')

            ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}\n'
                        f'(max_iter={max_iter})',
                        fontsize=16, fontweight='bold', pad=15)
            ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
            ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
            ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
            ax.set_xticks(np.arange(0.0, 0.11, 0.01))
            ax.set_yticks(np.arange(0, 101, 10))
            ax.tick_params(axis='both', labelsize=12)
            ax.set_ylim(-5, 105)
            ax.set_xlim(-0.005, 0.105)
            ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
            ax.legend(loc='upper left', fontsize=12)
            plt.tight_layout()

            plot_path = os.path.join(
                TRANSFER_RESULTS_DIR,
                f'{ATTACK_NAME}_iter{max_iter}_transferability_SEC.png'
            )
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.show()
            print(f"\n[INFO] SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Comparative overview — all max_iter, NN1 ASR vs NN2 ASR
        # --------------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(9, 6))

        ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                markeredgecolor='#2ca02c', markeredgewidth=1,
                markersize=6, linestyle='None', label='Baseline ASR (~0%)')

        for max_iter in MAX_ITER_LIST:
            if max_iter not in all_nn1_asrs:
                continue
            c_nn1 = ITER_COLORS_NN1[max_iter]
            c_nn2 = ITER_COLORS_NN2[max_iter]

            ax.plot([0.0, EPSILONS[0]], [0.0, all_nn1_asrs[max_iter][0]],
                    linestyle='-', color=c_nn1, linewidth=1.5)
            ax.plot(EPSILONS, all_nn1_asrs[max_iter],
                    marker='o', linestyle='-', color=c_nn1,
                    linewidth=1.5, markersize=5,
                    label=f'NN1 iter={max_iter}')

            ax.plot([0.0, EPSILONS[0]], [0.0, all_nn2_asrs[max_iter][0]],
                    linestyle='--', color=c_nn2, linewidth=1.5)
            ax.plot(EPSILONS, all_nn2_asrs[max_iter],
                    marker='s', linestyle='--', color=c_nn2,
                    linewidth=1.5, markersize=5,
                    label=f'NN2 iter={max_iter}')

        ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME} (Comparative)', fontsize=16, fontweight='bold', pad=15)
        ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
        ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
        ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
        ax.set_xticks(np.arange(0.0, 0.11, 0.01))
        ax.set_yticks(np.arange(0, 101, 10))
        ax.tick_params(axis='both', labelsize=12)
        ax.set_ylim(-5, 105)
        ax.set_xlim(-0.005, 0.105)
        ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
        ax.legend(loc='upper left', fontsize=9)
        plt.tight_layout()

        comparative_path = os.path.join(
            TRANSFER_RESULTS_DIR,
            f'{ATTACK_NAME}_transferability_comparative_SEC.png'
        )
        plt.savefig(comparative_path, dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n[INFO] Comparative SEC saved at: {comparative_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary
        # --------------------------------------------------------------------------
        print("\n" + "=" * 60)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY")
        print("=" * 60)
        for max_iter in MAX_ITER_LIST:
            if max_iter not in all_nn1_asrs:
                continue
            a1 = all_nn1_asrs[max_iter][-1]
            a2 = all_nn2_asrs[max_iter][-1]
            print(f"  max_iter={max_iter:2d} @ eps=0.1 — "
                f"NN1 ASR: {a1:.2f}%  NN2 ASR: {a2:.2f}%  "
                f"Gap: {(a2-a1):+.2f}%")
        print("=" * 60 + "\n")

KeyboardInterrupt: 

## Step 5.6: Transferability — PGD Targeted (Error-Specific)

Evaluates whether adversarial examples generated by PGD Targeted against NN1
cause NN2 to predict the same intended target class, across all combinations of
epsilon, max_iter, and num_random_init. Individual comparative targeted SECs are
generated for each (max_iter, num_random_init) combination. A summary table
reports the transferability gap for the strongest configuration.
Both targeting modes (Fixed and LeastLikely) are evaluated independently.

In [ ]:
if RUN_PGD_TRANSFERABILITY_SPECIFIC:

    # ==============================================================================
    # STEP 5.6: Transferability — PGD Targeted (Error-Specific)
    # ==============================================================================

    print("Evaluating PGD Targeted transferability on NN2...")

    ATTACK_NAME_BASE   = "PGD_Targeted"
    EPSILONS           = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
    MAX_ITER_LIST      = [1, 5, 10, 20, 40]
    NUM_RANDOM_INITS   = [0, 1, 3, 5]
    NUM_CLASSES        = 8631
    FIXED_TARGET_CLASS = 2026
    BATCH_SIZE         = 32
    TARGET_MODES       = ["LeastLikely", "Fixed"]

    ITER_COLORS_NN1 = {
        1: '#d62728', 5: '#8c1a1a', 10: '#e07070', 20: '#a83232', 40: '#6b0000'
    }
    ITER_COLORS_NN2 = {
        1: '#ff9896', 5: '#ffbcba', 10: '#ffd7d6', 20: '#ffe8e8', 40: '#fff0f0'
    }

    for mode in TARGET_MODES:
        ATTACK_NAME          = f"{ATTACK_NAME_BASE}_{mode}"
        TRANSFER_RESULTS_DIR = os.path.join(RESULTS_DIR, f"{ATTACK_NAME}_Transferability")
        os.makedirs(TRANSFER_RESULTS_DIR, exist_ok=True)

        nn1_results_path = os.path.join(RESULTS_DIR, ATTACK_NAME,
                                        f"{ATTACK_NAME}_results.csv")
        if not os.path.exists(nn1_results_path):
            print(f"[WARNING] NN1 results CSV not found for {ATTACK_NAME}. Skipping.")
            continue

        df_nn1 = pd.read_csv(nn1_results_path)

        print(f"\n{'*'*70}")
        print(f"Starting {ATTACK_NAME} transferability across "
                f"{len(NUM_RANDOM_INITS)} nri x "
                f"{len(MAX_ITER_LIST)} iters x "
                f"{len(EPSILONS)} epsilons")
        print(f"{'*'*70}")

        results            = []
        all_nn1_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}
        all_nn2_asrs       = {nri: {} for nri in NUM_RANDOM_INITS}

        for num_random_init in NUM_RANDOM_INITS:
            for max_iter in MAX_ITER_LIST:
                folder_name  = f"{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}"
                adv_iter_dir = os.path.join(ADVERSARIAL_DATA_DIR, folder_name)

                if not os.path.exists(adv_iter_dir):
                    print(f"[WARNING] Directory not found: {adv_iter_dir}. Skipping.")
                    continue

                print(f"\n{'='*50}")
                print(f"NUM_RANDOM_INIT = {num_random_init} | MAX_ITER = {max_iter}")
                print(f"{'='*50}")

                nn1_asrs = []
                nn2_asrs = []

                for eps in EPSILONS:
                    eps_folder  = f"eps_{eps:.3f}".replace('.', '_')
                    adv_eps_dir = os.path.join(adv_iter_dir, eps_folder)

                    if not os.path.exists(adv_eps_dir):
                        print(f"[WARNING] Directory not found: {adv_eps_dir}. Skipping.")
                        continue

                    print(f"\n--- eps={eps:.3f} | max_iter={max_iter} | nri={num_random_init} ---")

                    successful_nn2  = 0
                    total_processed = 0
                    start_time      = time.time()

                    for class_id in os.listdir(adv_eps_dir):
                        class_folder = os.path.join(adv_eps_dir, class_id)
                        if not os.path.isdir(class_folder):
                            continue

                        for img_name in os.listdir(class_folder):
                            if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                                continue

                            img_path = os.path.join(class_folder, img_name)
                            try:
                                img        = Image.open(img_path).convert('RGB')
                                tensor_nn2 = preprocess_nn2(img).unsqueeze(0).to(device)

                                with torch.no_grad():
                                    pred_idx_2 = nn2(tensor_nn2)[0].detach().cpu().numpy().argmax()

                                if mode == "Fixed":
                                    target = FIXED_TARGET_CLASS
                                else:
                                    clean_path = os.path.join(
                                        DATA_DIR_1000, class_id,
                                        img_name.replace('.png', '.jpg')
                                    )
                                    img_clean  = Image.open(clean_path).convert('RGB')
                                    t_clean    = preprocess_nn2(img_clean).unsqueeze(0).to(device)
                                    with torch.no_grad():
                                        preds_clean = nn2(t_clean)[0].detach().cpu().numpy()
                                    target = int(np.argmin(preds_clean))

                                if pred_idx_2 == target:
                                    successful_nn2 += 1

                                total_processed += 1

                            except Exception as e:
                                print(f"Error processing {img_path}: {e}")

                    elapsed = time.time() - start_time
                    asr_nn2 = (successful_nn2 / total_processed) * 100 if total_processed > 0 else 0.0

                    nn1_row = df_nn1[
                        (df_nn1['epsilon'] == eps) &
                        (df_nn1['max_iter'] == max_iter) &
                        (df_nn1['num_random_init'] == num_random_init)
                    ]
                    asr_nn1 = nn1_row['attack_success_rate'].values[0] if len(nn1_row) > 0 else 0.0

                    nn1_asrs.append(asr_nn1)
                    nn2_asrs.append(asr_nn2)

                    print(f"NN2 ASR: {asr_nn2:.2f}%  |  NN1 ASR: {asr_nn1:.2f}%")

                    results.append({
                        'attack_name':    ATTACK_NAME,
                        'target_mode':    mode,
                        'num_random_init': num_random_init,
                        'max_iter':       max_iter,
                        'epsilon':        eps,
                        'nn1_asr':        round(asr_nn1, 2),
                        'nn2_asr':        round(asr_nn2, 2),
                        'nn2_successful': successful_nn2,
                        'total_images':   total_processed,
                        'time_elapsed_s': round(elapsed, 2)
                    })

                all_nn1_asrs[num_random_init][max_iter] = nn1_asrs
                all_nn2_asrs[num_random_init][max_iter] = nn2_asrs

                # ------------------------------------------------------------------
                # Individual Comparative Targeted SEC for this (nri, max_iter)
                # ------------------------------------------------------------------
                c_nn1 = ITER_COLORS_NN1[max_iter]
                c_nn2 = ITER_COLORS_NN2[max_iter]

                fig, ax = plt.subplots(figsize=(9, 6))

                ax.plot(0.0, 0.0, marker='o', markerfacecolor='gray',
                        markeredgecolor='#2ca02c', markeredgewidth=1,
                        markersize=6, linestyle='None', label='Baseline ASR (~0%)')

                ax.plot([0.0, EPSILONS[0]], [0.0, nn1_asrs[0]],
                        linestyle='-', color=c_nn1, linewidth=1.5)
                ax.plot(EPSILONS, nn1_asrs,
                        marker='o', linestyle='-', color=c_nn1,
                        linewidth=1.5, markersize=6,
                        label=f'NN1 ASR (nri={num_random_init}, iter={max_iter})')

                ax.plot([0.0, EPSILONS[0]], [0.0, nn2_asrs[0]],
                        linestyle='--', color=c_nn2, linewidth=1.5)
                ax.plot(EPSILONS, nn2_asrs,
                        marker='s', linestyle='--', color=c_nn2,
                        linewidth=1.5, markersize=6,
                        label=f'NN2 ASR (nri={num_random_init}, iter={max_iter})')

                ax.set_title(f'Transferability Targeted SEC — {ATTACK_NAME}\n'
                            f'(nri={num_random_init}, max_iter={max_iter})',
                            fontsize=16, fontweight='bold', pad=15)
                ax.set_xlabel(r'Perturbation Budget ($\epsilon$ — $L_\infty$ Norm)', fontsize=14)
                ax.set_ylabel('Attack Success Rate (%)', fontsize=14)
                ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
                ax.set_xticks(np.arange(0.0, 0.11, 0.01))
                ax.set_yticks(np.arange(0, 101, 10))
                ax.tick_params(axis='both', labelsize=12)
                ax.set_ylim(-5, 105)
                ax.set_xlim(-0.005, 0.105)
                ax.axvline(x=0.1, color='black', linestyle=':', linewidth=2, label=r'Max Allowable $\epsilon$ (0.1)')
                ax.legend(loc='upper left', fontsize=11)
                plt.tight_layout()

                plot_path = os.path.join(
                    TRANSFER_RESULTS_DIR,
                    f'{ATTACK_NAME}_nri{num_random_init}_iter{max_iter}_transferability_SEC.png'
                )
                plt.savefig(plot_path, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"\n[INFO] SEC saved at: {plot_path}")

        # --------------------------------------------------------------------------
        # Save Results CSV
        # --------------------------------------------------------------------------
        df_results = pd.DataFrame(results)
        csv_path   = os.path.join(TRANSFER_RESULTS_DIR, f'{ATTACK_NAME}_transferability_results.csv')
        df_results.to_csv(csv_path, index=False)
        print(f"\n[INFO] Results CSV saved at: {csv_path}")

        # --------------------------------------------------------------------------
        # Summary — strongest configuration only
        # --------------------------------------------------------------------------
        best_nri  = max(NUM_RANDOM_INITS)
        best_iter = max(MAX_ITER_LIST)

        print("\n" + "=" * 60)
        print(f"{ATTACK_NAME} TRANSFERABILITY SUMMARY " f"(Strongest Config: nri={best_nri}, iter={best_iter})")
        print("=" * 60)
        if (best_nri in all_nn1_asrs and
                best_iter in all_nn1_asrs[best_nri]):
            a1 = all_nn1_asrs[best_nri][best_iter][-1]
            a2 = all_nn2_asrs[best_nri][best_iter][-1]
            print(f"  @ eps=0.1 — NN1 ASR: {a1:.2f}%  " f"NN2 ASR: {a2:.2f}%  Gap: {(a2-a1):+.2f}%")
        print("=" * 60 + "\n")